# Structural model of a haploid *S. cerevisiae* cell (after Maritan et al. 2021, JMB 167351)
Layer 1: composition inventory.

In [2]:
import sys, json, importlib; sys.path.insert(0,'/workspace/yc')
import numpy as np, pandas as pd
import inventory as inv; importlib.reload(inv)
df, geo = inv.build()
print('PaxDb ppm matched to UniProt ORFs:', round(df.attrs['paxdb_ppm_matched']), 'of 1e6')
print('proteins with abundance >0:', (df.copies_per_cell>0).sum(), '/', len(df))
print('loc source:', df.loc_source.value_counts().to_dict())
print(json.dumps({k:(round(v,3) if isinstance(v,float) else v) for k,v in geo.items() if not isinstance(v,dict)}))
print({k:round(v,3) for k,v in geo['volumes_um3'].items()})
print({k:round(v,2) for k,v in geo['areas_um2'].items()})
comp = inv.per_compartment(df, geo); print(comp.round(1).to_string(index=False))
print(df.head(15)[['gene','copies_per_cell','compartments','oligomer_n','mass_da']].to_string())

PaxDb ppm matched to UniProt ORFs: 981894 of 1e6
proteins with abundance >0: 5832 / 6067
loc source: {'UniProt': 4457, 'GO': 985, 'default_cytosol': 625}
{"cell_radius_um": 2.156, "protoplast_radius_um": 2.046, "nucleus_radius_um": 0.889, "n_npc": 117, "cer_cover": 0.48, "wall_thickness_um": 0.11}
{'cell_wall': 6.105, 'nucleolus': 0.588, 'nucleoplasm': 2.352, 'vacuole': 2.94, 'mitochondrion': 0.42, 'lipid_droplet': 0.42, 'golgi_endosome': 0.336, 'peroxisome': 0.084, 'ER': 0.904, 'nuclear_envelope': 0.0, 'plasma_membrane': 0.0, 'cytosol': 27.851}
{'plasma_membrane': 52.62, 'nuclear_envelope': 19.85, 'ER': 50.52, 'vacuole': 9.92}
     compartment  protein_copies  protein_mass_da  volume_um3  area_um2  protein_mg_per_ml
         cytosol      32299221.5     1.179170e+12        27.9       NaN               70.3
   mitochondrion       3255467.4     1.362872e+11         0.4       NaN              538.8
     nucleoplasm       1939766.7     8.181876e+10         2.4       NaN               57.8


In [4]:
import ingredients as ingm; importlib.reload(ingm)
ing, consumed = ingm.build_ingredients(df)
print(ing[ing.kind=='complex'][['ing_id','copies_per_cell','n_subunit_types','mass_da','pdb']].round(0).to_string(index=False))
print('residual protein ingredients:', (ing.kind=='protein').sum(), ' residual protein copies:', round(ing[ing.kind=='protein'].copies_per_cell.sum()))

        ing_id  copies_per_cell  n_subunit_types   mass_da  pdb
  ribosome_80S         169002.0               78 1406524.0 4V88
proteasome_26S           7657.0               33 1307668.0 4CR2
           FAS           8669.0                2 2613828.0 2UV8
           CCT           4994.0                8  954676.0 4V94
         PolII           5411.0               12  513531.0 1WCM
          PolI           7721.0               14  589502.0 4C2M
        PolIII           2267.0               17  693209.0 5FJ8
           NPC            117.0               33 3284255.0 7N9F
  Pma1_hexamer          41477.0                1  605373.0 7VH5
      V-ATPase          11720.0               14 1102112.0 6O7V
    nucleosome          73152.0                4   56467.0 1ID3
residual protein ingredients: 5703  residual protein copies: 29134444


In [5]:
import build_structs as bs, structlib as sl; importlib.reload(sl); importlib.reload(bs)
ing, consumed = ingm.build_ingredients(df)
topo = pd.read_csv('/workspace/yc/data/uniprot_topology.tsv', sep='\t'); topo.columns=['acc','topo','tm','intramem']
topo = topo.set_index('acc')
pma1_acc = df.set_index('gene').loc['PMA1','uniprot']
pma1_tm = bs.parse_ranges(topo.loc[pma1_acc,'tm'], 'TRANSMEM')
struct_info = {}
for r in ing[ing.kind=='complex'].itertuples():
    struct_info[r.ing_id] = ('PDB:'+r.pdb,) + bs.process_complex(r.ing_id, r.pdb, pma1_tm)
    print(r.ing_id, struct_info[r.ing_id])
# homo-oligomers among the most abundant residual soluble proteins
cand = ing[(ing.kind=='protein') & (ing.oligomer_n>1) & (ing.n_tm==0)].sort_values('copies_per_cell', ascending=False).head(40)
homo = {}
for r in cand.itertuples():
    pid = sl.rcsb_homo_assembly(r.uniprot, int(r.oligomer_n))
    homo[r.ing_id] = pid
print({k:v for k,v in homo.items()})

ribosome_80S ('PDB:4V88', 16507, 413, 16.100407364296487)
proteasome_26S ('PDB:4CR2', 10166, 255, 14.80287740767197)


FAS ('PDB:2UV8', 21876, 547, 14.798600461067087)
CCT ('PDB:4V94', 8426, 211, 9.018708223321946)
PolII ('PDB:1WCM', 3902, 98, 9.91034382583473)


PolI ('PDB:4C2M', 4345, 109, 10.219339801718824)
PolIII ('PDB:5FJ8', 4886, 123, 10.34536498647441)


NPC ('PDB:7N9F', 216568, 5415, 64.6805638635999)
Pma1_hexamer ('PDB:7VH5', 4824, 121, 11.32574556943074)
V-ATPase ('PDB:6O7V', 8038, 201, 19.620152035235616)


nucleosome ('PDB:1ID3', 1057, 27, 6.2679600459814155)


{'TDH3': '3PYM', 'FBA1': None, 'ENO2': None, 'PDC1': '1QPB', 'CDC19': '1A3W', 'ADH1': '4W6Z', 'TPI1': '1NEY', 'GPM1': '1QHF', 'TDH2': None, 'ENO1': '2AL1', 'AHP1': '4H86', 'TSA1': None, 'TDH1': None, 'SOD1': '1F1G', 'PGI1': None, 'HYP2': None, 'HSP82': '2WER', 'HXK2': None, 'IPP1': '1E9G', 'GND1': '2P4Q', 'BMH1': None, 'URA1': None, 'GLN1': None, 'TRR1': '3ITJ', 'GDH1': None, 'TAL1': None, 'HOM6': '1EBF', 'ADH2': None, 'YDJ1': '1XAO', 'ARG1': None, 'SHM2': None, 'CYS3': '1N8P', 'TKL1': '1GPU', 'KRS1': '7F6W', 'HOM2': None, 'GLY1': None, 'SES1': None, 'PDC5': None, 'GRS1': None, 'SEC53': None}


In [6]:
importlib.reload(bs)
import time; t0=time.time()
af_res, af_missing = bs.process_all_af(ing, topo)
print('AF processed', len(af_res), 'missing', len(af_missing), 'errors', sum(1 for v in af_res.values() if v[0]=='ERR'), f'{time.time()-t0:.0f}s')
# homo-oligomer PDB assemblies (with paralog substitution) replace AF monomers for abundant enzymes
paralog_sub = {'TDH1':'TDH3','TDH2':'TDH3','ENO2':'ENO1','ADH2':'ADH1','PDC5':'PDC1'}
homo_pdb = {k:v for k,v in homo.items() if v}
for a,b in paralog_sub.items():
    if homo.get(b): homo_pdb[a] = homo[b]
oligo = {}
for g, pid in homo_pdb.items():
    acc = ing.set_index('ing_id').loc[g,'uniprot']
    oligo[g+'_oligo'] = (g, pid, int(ing.set_index('ing_id').loc[g,'oligomer_n'])) + bs.process_homo_pdb(g+'_oligo', pid, acc)
print(pd.DataFrame(oligo, index=['gene','pdb','n','beads','spheres','rmax']).T.to_string())

AF processed 5692 missing 11 errors 0 21s


              gene   pdb  n beads spheres      rmax
TDH3_oligo    TDH3  3PYM  4  1326      45   4.75245
PDC1_oligo    PDC1  1QPB  4  2220      74  6.538005
CDC19_oligo  CDC19  1A3W  4  1962      66  6.719529
ADH1_oligo    ADH1  4W6Z  4  1388      47  5.133056
TPI1_oligo    TPI1  1NEY  2   492      17  3.759819
GPM1_oligo    GPM1  1QHF  4   960      32  4.910197
ENO1_oligo    ENO1  2AL1  2   867      29  4.253678
AHP1_oligo    AHP1  4H86  2   374      13  4.108015
SOD1_oligo    SOD1  1F1G  2   306      11  3.453137
HSP82_oligo  HSP82  2WER  2   428      15  4.246291
IPP1_oligo    IPP1  1E9G  2   567      19  4.492556
GND1_oligo    GND1  2P4Q  2   952      32  4.982699
TRR1_oligo    TRR1  3ITJ  2   624      21  4.726641
HOM6_oligo    HOM6  1EBF  2   716      24  4.285758
YDJ1_oligo    YDJ1  1XAO  2   230       8  3.469291
CYS3_oligo    CYS3  1N8P  4  1572      53  5.142821
TKL1_oligo    TKL1  1GPU  2  1356      46  4.804234
KRS1_oligo    KRS1  7F6W  2  1012      34  5.402975
TDH1_oligo  

In [7]:
importlib.reload(inv); import ingredients as ingm; importlib.reload(ingm)
df, geo = inv.build()
comp = inv.per_compartment(df, geo)
ing, consumed = ingm.build_ingredients(df)
print(comp.to_string())
print(ing.columns.tolist())
print({k:(v if not isinstance(v,dict) else {kk:round(vv,3) if isinstance(vv,float) else vv for kk,vv in v.items()}) for k,v in geo.items()})
print(ing[ing.kind!='protein'][['ing_id','copies_per_cell','mass_da']].to_string())

         compartment  protein_copies  protein_mass_da  volume_um3   area_um2  protein_mg_per_ml
0            cytosol    3.287924e+07     1.199431e+12   27.430840        NaN          72.609827
2      mitochondrion    2.832293e+06     1.219673e+11    0.840000        NaN         241.114515
1        nucleoplasm    1.960115e+06     8.259946e+10    2.352000        NaN          58.317534
5          nucleolus    1.131517e+06     4.656860e+10    0.588000        NaN         131.514956
7                 ER    8.422245e+05     4.187836e+10    0.903951  50.517632          76.931427
4    plasma_membrane    7.389495e+05     4.814163e+10    0.000000  52.622533                inf
8     golgi_endosome    4.490757e+05     2.254523e+10    0.336000        NaN         111.422964
6            vacuole    4.437440e+05     2.575710e+10    2.940000   9.924661          14.548189
3          cell_wall    3.888635e+05     1.874105e+10    6.105208        NaN           5.097445
9         peroxisome    1.284582e+05    

In [8]:
import column as col; importlib.reload(col)
exp = col.expected_counts(ing, geo)
print(exp.groupby('region').expected.agg(['sum','count']).round(1).to_string())
print(exp.sort_values('expected', ascending=False).head(15).to_string())
npc = np.load('/workspace/yc/lib/NPC.npz'); print('NPC bead extent (nm) x,y,z:', np.ptp(npc['beads'],axis=0).round(1))

                   sum  count
region                       
ER_lumen         343.8    115
ER_membrane      704.4    385
NE               128.9     41
NE_peripheral    544.8    100
PM               287.9    231
PM_peripheral    343.4    136
cell_wall        437.9    128
cytosol        22484.3   2523
nucleoplasm     5211.6   1682
          ing_id   region     expected
4147        TDH3  cytosol  1049.400852
1190        FBA1  cytosol   853.833975
1087        ENO2  cytosol   715.845163
2743        PGK1  cytosol   621.220396
538        CDC19  cytosol   575.645282
2675        PDC1  cytosol   541.182463
59          ADH1  cytosol   477.455728
3275       RPL39  cytosol   462.970964
4275        TPI1  cytosol   405.257527
4152  TEF1; TEF2  cytosol   390.682515
1452        GPM1  cytosol   341.046252
4146        TDH2  cytosol   325.433391
1086        ENO1  cytosol   324.937029
96          AHP1  cytosol   321.236871
4743        YEF3  cytosol   279.091171
NPC bead extent (nm) x,y,z: [124.7 124.7  52.9

In [9]:
rp = ing[(ing.kind=='protein') & ing.ing_id.str.match(r'^RP[LS]\d')]
print(len(rp), 'residual RP ingredients; copies', round(rp.copies_per_cell.sum()), '; mass fraction of all protein', round((rp.copies_per_cell*rp.mass_da).sum()/(ing.copies_per_cell*ing.mass_da).sum(),4))
print(rp.sort_values('copies_per_cell',ascending=False).head(8)[['ing_id','copies_per_cell']].round(0).to_string())

70 residual RP ingredients; copies 2126233 ; mass fraction of all protein 0.0224
    ing_id  copies_per_cell
14   RPL39         438866.0
25   RPL10         128580.0
37  RPL35B          79397.0
40  RPL35A          75147.0
38  RPL24A          69347.0
44   RPL8B          62032.0
53  RPL24B          58163.0
47  RPS11B          57537.0


In [16]:
# excess free ribosomal-protein pool (MS noise above the ribosome count) is not packed; documented as an assumption
ing_pack = ing[~((ing.kind=='protein') & ing.ing_id.str.match(r'^RP[LS]\d'))].copy()
oligo_map = {v[0]: (k, v[2]) for k, v in oligo.items()}
importlib.reload(col)
t0 = time.time()
pk, cexp, clog, linker, clib = col.build_column(ing_pack, geo, oligo_map, seed=7)
# RNA: tRNA (yeast tRNA-Phe, PDB 1EHZ) at 3.3e6 per cell (Waldron & Lacroute 1975 via Shah et al. 2013), cytosolic
import gemmi, structlib as sl
st = gemmi.read_structure('/workspace/yc/pdb/1EHZ-assembly1.cif.gz'); st.remove_ligands_and_waters()
xyz, knd, _ = sl.beads_from_structure(st); xyz = xyz - xyz.mean(0)
bs.save('tRNA', xyz, knd, per=15); clib['tRNA'] = dict(np.load('/workspace/yc/lib/tRNA.npz'))
N_TRNA, N_MRNA = 3.3e6, 3.0e4
rv = col.region_volumes(); fcol = rv['cytosol'] / (geo['volumes_um3']['cytosol'] * 1e9)
rng_r = np.random.default_rng(5)
n_t = col.stochastic_round(N_TRNA * fcol, rng_r)
ok_t = pk.try_place('tRNA', clib['tRNA']['st_c'], clib['tRNA']['st_r'], col.regions()['cytosol'], n_t, max_tries=300)
pk.tags += ['cytosol'] * ok_t
clog = pd.concat([clog, pd.DataFrame([('tRNA', 'cytosol', n_t, ok_t)], columns=clog.columns)], ignore_index=True)
print(f'packed in {time.time()-t0:.0f}s; objects placed {len(pk.placed)}; spheres {pk.g.n_sph[0]}; tRNA beads {len(xyz)}')
s = clog.groupby('region')[['requested','placed']].sum(); s['frac']=s.placed/s.requested
print(s.round(3).to_string())
print('requested without structure:', cexp[~cexp.has_struct].n.sum(), 'of', cexp.n.sum())

packed in 5s; objects placed 27621; spheres 500814; tRNA beads 73
               requested  placed   frac
region                                 
ER_lumen             335     312  0.931
ER_membrane_A        369     369  1.000
ER_membrane_B        344     344  1.000
NE_INM                66      66  1.000
NE_ONM                59      59  1.000
NE_peripheral        540     502  0.930
PM                   282     282  1.000
PM_peripheral        342     340  0.994
cell_wall            436     436  1.000
cytosol            19826   19826  1.000
nucleoplasm         5091    5084  0.999
requested without structure: 11 of 24220


In [17]:
import assemble as asm; importlib.reload(asm)
from scipy.spatial import cKDTree
rng = np.random.default_rng(11)
t0=time.time()
ppos, pinst, pkind = asm.expand(pk, clib)
ptree = cKDTree(ppos)
print('protein/NA residue beads', len(ppos), f'{time.time()-t0:.0f}s')
lip = asm.lipids(rng, ptree)
print('lipids placed', lip['n_lipids_total']-lip['n_removed'], 'removed for protein clash', lip['n_removed'], f'{time.time()-t0:.0f}s')
tot_prot_da = (ing.copies_per_cell*ing.mass_da).sum()
bud = asm.wall_mass_budget(tot_prot_da, geo)
print({k: (f'{v:.3g}') for k,v in bud.items()})
wall_sel = np.isin(pinst, [i for i,t in enumerate(pk.tags) if t=='cell_wall'])
glc, glc_fid = asm.glucan(rng, bud, ptree)
man, man_owner, man_target = asm.mannan(rng, bud, ppos[wall_sel], pinst[wall_sel], ptree)
print('glucan beads', len(glc), 'fibrils', glc_fid.max()+1, '| mannan beads', len(man), 'target', man_target, f'{time.time()-t0:.0f}s')
n_m = col.stochastic_round(N_MRNA * fcol, rng)
ltree = cKDTree(lip['pos'])
mrna = asm.mrna_threads(rng, n_m, ptree, col.LAY['pm'] + 3, col.LAY['onm'] - 3)
print('mRNA chains', n_m, 'nt placed', sum(len(c) for c in mrna), f'{time.time()-t0:.0f}s')

protein/NA residue beads 15449736 5s


lipids placed 693562 removed for protein clash 101454 10s
{'dry_mass_pg': '6.11', 'wall_mass_pg': '1.22', 'column_fraction': '0.00113', 'b13_glucose': '2.68e+06', 'b16_glucose': '3.83e+05', 'mannose': '1.63e+06', 'chitin_glcnac': '6.12e+04'}


glucan beads 85267 fibrils 568 | mannan beads 438093 target 543084 13s


mRNA chains 31 nt placed 46445 15s


In [18]:
import render as rd, style as sty; importlib.reload(rd); importlib.reload(sty)
print('nucleosomes placed:', clog[clog.lib_id=='nucleosome'][['requested','placed']].values.tolist(), '| linker beads', len(linker))
S_pos, S_rad, S_col, S_mol, lip_range = sty.assemble_scene(pk, ppos, pinst, pkind, lip, glc, glc_fid, man, man_owner, linker, np.random.default_rng(3), mrna=mrna)
print('scene beads', len(S_pos))
os.makedirs('/workspace/yc/fig', exist_ok=True)
ycut, depth = 100.0, 30.0
t0=time.time()
full = rd.render(S_pos, S_rad, S_col, S_mol, bounds=((-30, 760), (0, 250)), px_per_nm=6, ycut=ycut, depth=depth, ss=2, bg_fn=sty.make_bg(ycut+depth/2), no_inner=lip_range)
full.save('/workspace/yc/fig/column_full.png'); print(full.size, f'{time.time()-t0:.0f}s')
full.crop((0, 0, int(260*6), 250*6)).save('/workspace/yc/fig/crop_wall.png')
full.crop((int(530*6), 0, int(790*6), 250*6)).save('/workspace/yc/fig/crop_nucleus.png')

nucleosomes placed: [[256, 256]] | linker beads 2500


scene beads 18018863


(4740, 1500) 11s


In [24]:
import wholecell as wc; importlib.reload(wc)
t0 = time.time(); rng_w = np.random.default_rng(21)
lab, ax_w, shp = wc.build_layout(geo, rng_w)
mpix = wc.membrane_pixels(lab); edt = wc.edt_by_label(lab)
print('layout', lab.shape, f'{time.time()-t0:.0f}s', 'NPCs in section', len(shp['npc']))
objs, slab_vol, mlen, memb_frac = wc.make_objects(ing_pack, oligo_map, geo, lab, mpix, N_TRNA, rng_w)
print('objects requested', len(objs))
WO = wc.pack(objs, lab, mpix, edt, seed=3, max_tries=80)
print(f'packed {WO.ok.sum()} / {len(WO)} in {time.time()-t0:.0f}s')
summ = WO.assign(region=[m if isinstance(m,str) else wc.LABEL_NAMES[c] for m,c in zip(WO.memb, WO.comp)]).groupby('region').ok.agg(['size','sum','mean'])
print(summ.round(3).to_string())
# area fractions of the section vs volume fractions
ar = np.bincount(lab.ravel(), minlength=15); inside = ar[wc.CYTO:].sum()
print({wc.LABEL_NAMES[l]: round(ar[l]/inside,4) for l in range(3,15)})

layout (4434, 4434) 139s NPCs in section 9


objects requested 261184


packed 260652 / 261184 in 141s
               size     sum   mean
region                            
INM             139     139  1.000
LD              112     112  1.000
MIM            5064    4632  0.915
ONM             143     143  1.000
PM             1819    1819  1.000
cER            3688    3688  1.000
cER_lumen       814     803  0.986
cytosol      155547  155547  1.000
golgi          3206    3206  1.000
mito_matrix   28369   28280  0.997
nucleolus     20166   20166  1.000
nucleoplasm   32460   32460  1.000
perox           217     217  1.000
peroxisome      931     931  1.000
tonoplast      1804    1804  1.000
vacuole        3934    3934  1.000
wall_mannan    2771    2771  1.000


{'cytosol': np.float64(0.5731), 'cER_lumen': np.float64(0.0067), 'perinuclear': np.float64(0.0121), 'nucleoplasm': np.float64(0.1335), 'nucleolus': np.float64(0.0408), 'vacuole': np.float64(0.1872), 'mito_IMS': np.float64(0.0034), 'mito_matrix': np.float64(0.0217), 'lipid_droplet': np.float64(0.011), 'golgi': np.float64(0.006), 'peroxisome': np.float64(0.0019), 'pore': np.float64(0.0025)}


In [25]:
importlib.reload(wc); importlib.reload(rd)
W = WO[WO.ok].reset_index(drop=True)
rng_c = np.random.default_rng(8)
Wcol = np.array([wc.obj_color(o, rng_c) for o in W.itertuples()])
# NPC cross-sections: cytoplasmic, inner and nuclear rings on both sides of each pore + basket
npc_pts = []
for p, a in shp['npc']:
    n_ = np.array([np.cos(a), np.sin(a)]); t_ = np.array([-n_[1], n_[0]])
    for s in (-1, 1):
        npc_pts += [(p + s*t_*52, 12), (p + n_*22 + s*t_*58, 10), (p - n_*22 + s*t_*58, 10)]
        for k in range(1, 5): npc_pts.append((p - n_*(22 + 12*k) + s*t_*(58 - 11*k), 4.5))
    npc_pts.append((p - n_*72, 6))
NP = np.array([q[0] for q in npc_pts]); NR = np.array([q[1] for q in npc_pts])
npc_col = np.tile(rd.hex2rgb(wc.WPAL['NPC']), (len(NP), 1))
posW = np.c_[np.r_[W.v, NP[:,1]], np.r_[W.z + 15, np.full(len(NP), 15.0)], np.r_[W.u, NP[:,0]]]
radW = np.r_[W.r, NR]; colW = np.vstack([Wcol, npc_col]); molW = np.arange(len(radW))
bgimg = wc.background_image(lab, mpix, width=2)
half = lab.shape[0] / 2
def bgW(zz, xx):
    j = np.clip((zz + half).astype(int), 0, lab.shape[1]-1); i = np.clip((xx + half).astype(int), 0, lab.shape[0]-1)
    return bgimg[i, j]
t0 = time.time()
R = shp['Rcell'] + 40
cell_im = rd.render(posW, radW, colW, molW, bounds=((-R, R), (-R, R)), px_per_nm=1.0, ycut=0, depth=30, ss=2, bg_fn=bgW, fog_strength=0.3, dz_in=50, outline_alpha=0.4)
cell_im.save('/workspace/yc/fig/wholecell.png'); print(cell_im.size, f'{time.time()-t0:.0f}s')
# zoom: cortex + mitochondrion + nucleus edge at 3 px/nm
zoom = rd.render(posW, radW, colW, molW, bounds=((-1700, -900), (-1150, -550)), px_per_nm=3.0, ycut=0, depth=30, ss=2, bg_fn=bgW, fog_strength=0.3, dz_in=50)
zoom.save('/workspace/yc/fig/wholecell_zoom.png'); print(zoom.size)

(4392, 4392) 19s


(2400, 1800)


In [26]:
zoom = rd.render(posW, radW, colW, molW, bounds=((-2200, -1400), (150, 750)), px_per_nm=3.0, ycut=0, depth=30, ss=2, bg_fn=bgW, fog_strength=0.3, dz_in=50)
zoom.save('/workspace/yc/fig/wholecell_zoom.png'); print(zoom.size)

(2400, 1800)


In [27]:
# ---------- validation ----------
V = {}
V['total_protein_copies'] = df.copies_per_cell.sum()
V['total_protein_mass_pg'] = (df.copies_per_cell*df.mass_da).sum()*1.66054e-12
V['frac_ppm_matched'] = 0.981894
# 1) packed vs expected (column): Poisson z-scores per ingredient-region, using integer requests drawn from expectations
ce = cexp.groupby(['lib_id','region'], as_index=False)[['expected','n']].sum()
pl = pd.Series([p[0] for p in pk.placed]).value_counts()
libreg = clog.groupby('lib_id')[['requested','placed']].sum()
ce_l = ce.groupby('lib_id')[['expected','n']].sum().join(libreg, how='left').fillna(0)
ce_l = ce_l[ce_l.expected>0]
ce_l['z'] = (ce_l.placed - ce_l.expected)/np.sqrt(ce_l.expected)
V['column_expected_total'] = ce_l.expected.sum(); V['column_placed_total'] = ce_l.placed.sum()
V['column_frac_ing_|z|<2'] = (ce_l.z.abs()<2).mean()
V['column_frac_ing_|z|<3'] = (ce_l.z.abs()<3).mean()
# 2) residue-level clash check between different instances (random sample of 300k beads)
rs = np.random.default_rng(0); idx = rs.choice(len(ppos), 300000, replace=False)
nb_ = ptree.query_ball_point(ppos[idx], r=0.3)
clash = sum(any(pinst[j]!=pinst[i] for j in js) for i, js in zip(idx, nb_))
V['bead_clash_rate_<0.3nm'] = clash/len(idx)
# 3) macromolecular crowding in column compartments (protein + RNA mass from structures)
mass_by_inst = {}
libmass = {**dict(zip(ing.ing_id, ing.mass_da)), **{k: v[2]*ing.set_index('ing_id').loc[v[0],'mass_da'] for k,v in oligo.items()}, 'tRNA': 25000, 'ribosome_80S': 1.4065e6+1.8e6, 'nucleosome': 1.099e5+147*650}
reg_mass = {}
for (lid,_,_), tag in zip(pk.placed, pk.tags):
    reg_mass[tag] = reg_mass.get(tag,0) + libmass.get(lid,0)
rvol = col.region_volumes()
for t, vol in [('cytosol', rvol['cytosol']), ('nucleoplasm', rvol['nucleoplasm'])]:
    mgml = reg_mass[t]/6.022e23/(vol*1e-21)*1e3
    V[f'column_{t}_macromolecule_mg_per_ml'] = mgml
    V[f'column_{t}_volume_fraction'] = reg_mass[t]/813/vol
# 4) fraction of placed mass with experimental vs predicted structures
src = {}
exp_ids = set(['ribosome_80S','proteasome_26S','FAS','CCT','PolI','PolII','PolIII','NPC','Pma1_hexamer','V-ATPase','nucleosome','tRNA']) | set(oligo)
for (lid,_,_) in pk.placed:
    k = 'experimental (PDB)' if lid in exp_ids else 'AlphaFold model'
    src[k] = src.get(k,0) + libmass.get(lid,0)
tot = sum(src.values()); V.update({f'mass_frac_{k}': v/tot for k,v in src.items()})
# AF pLDDT trimming
kept = [float(np.load(f'/workspace/yc/lib/{l}.npz')['plddt_frac_kept']) for l in set(p[0] for p in pk.placed) if l not in exp_ids]
V['AF_median_frac_residues_kept_pLDDT>=50'] = float(np.median(kept))
# 5) whole-cell slab
V['wholecell_objects_requested'] = len(WO); V['wholecell_objects_placed'] = int(WO.ok.sum())
V['lipids_in_column'] = int(lip['n_lipids_total']-lip['n_removed']); V['membrane_area_fraction_protein_PM_column'] = lip['n_removed']/lip['n_lipids_total']
for k,v in V.items(): print(f'{k:50s} {v:.4g}' if isinstance(v,(float,np.floating)) else f'{k:50s} {v}')

total_protein_copies                               4.2e+07
total_protein_mass_pg                              2.698
frac_ppm_matched                                   0.9819
column_expected_total                              2.421e+04
column_placed_total                                2.414e+04
column_frac_ing_|z|<2                              0.9837
column_frac_ing_|z|<3                              0.9936
bead_clash_rate_<0.3nm                             3.333e-06
column_cytosol_macromolecule_mg_per_ml             94.29
column_cytosol_volume_fraction                     0.06984
column_nucleoplasm_macromolecule_mg_per_ml         55.24
column_nucleoplasm_volume_fraction                 0.04092
mass_frac_experimental (PDB)                       0.4934
mass_frac_AlphaFold model                          0.5066
AF_median_frac_residues_kept_pLDDT>=50             0.8966
wholecell_objects_requested                        261184
wholecell_objects_placed                           260652
lipid

In [28]:
import figs; importlib.reload(figs)
os.makedirs('/mnt/results/figures', exist_ok=True)
col_labels = [(0,35,'mannoprotein layer'), (35,110,'β-glucan wall'), (108,119,'PM'), (138,166,'cortical ER'), (175,585,'cytoplasm'), (588,630,'nuclear envelope + pore'), (632,760,'nucleoplasm')]
figs.annotate_column('/workspace/yc/fig/column_full.png', '/mnt/results/figures/fig1_column_goodsell', labels=col_labels)
R = shp['Rcell'] + 40
P = lambda u, v: (u + R, v + R)
cn, rn = shp['nucleus']; cv, rv_ = shp['vacuole']
p0 = shp['npc'][0]
labels = [(*P(cn[0]+250, cn[1]-250), 'nucleoplasm'), (*P(*(shp['nucleolus_c'] - (shp['nucleolus_c']-cn)*0.55)), 'nucleolus'),
          (*P(*cv), 'vacuole'), (*P(90, -1375), 'mitochondrion'), (*P(-1250, -820), 'mitochondrion'),
          (*P(-300, -900), 'lipid droplet'), (*P(700, 1180), 'Golgi / endosomes'), (*P(1500, 700), 'peroxisome'),
          (*P(0, -2105), 'cell wall'), (*P(-300, 1500), 'cytoplasm')]
figs.annotate_cell('/workspace/yc/fig/wholecell.png', '/workspace/yc/fig/wholecell_zoom.png', '/mnt/results/figures/fig2_wholecell_goodsell', labels,
                   zoom_box_px=(-2200+R, 150+R, -1400+R, 750+R))
print(os.listdir('/mnt/results/figures'))

['fig1_column_goodsell.png', 'fig1_column_goodsell.svg', 'fig2_wholecell_goodsell.png', 'fig2_wholecell_goodsell.svg']


In [29]:
import matplotlib.pyplot as plt, seaborn as sns
importlib.reload(figs)
figs.annotate_column('/workspace/yc/fig/column_full.png', '/mnt/results/figures/fig1_column_goodsell', labels=col_labels)
figs.annotate_cell('/workspace/yc/fig/wholecell.png', '/workspace/yc/fig/wholecell_zoom.png', '/mnt/results/figures/fig2_wholecell_goodsell', labels, zoom_box_px=(-2200+R, 150+R, -1400+R, 750+R))
fig, axs = plt.subplots(1, 3, figsize=(12, 3.8), dpi=200)
# A: protein concentration by compartment
c = comp[comp.volume_um3 > 0].copy().sort_values('protein_mg_per_ml')
c = c[np.isfinite(c.protein_mg_per_ml)]
cb = sns.color_palette('colorblind')
axs[0].barh(c.compartment.str.replace('_',' '), c.protein_mg_per_ml, color=cb[0])
for y, (v, n) in enumerate(zip(c.protein_mg_per_ml, c.protein_copies)):
    axs[0].text(v + 4, y, f'{n/1e6:.2g}M', va='center', fontsize=7)
axs[0].set_xlabel('Protein concentration (mg/mL)'); axs[0].set_title('A  Proteome by compartment', loc='left', fontsize=10)
axs[0].set_xlim(0, c.protein_mg_per_ml.max()*1.25)
# B: packed vs expected (column, per ingredient); zero placements drawn at 0.5
m = ce_l[ce_l.expected > 0.05]
axs[1].scatter(m.expected, m.placed + 0.5*(m.placed==0), s=6, alpha=0.5, color=cb[0], lw=0)
lim = [0.04, m.expected.max()*1.5]; xs = np.logspace(np.log10(lim[0]), np.log10(lim[1]), 100)
axs[1].plot(xs, xs, color='k', lw=0.8); axs[1].fill_between(xs, np.maximum(xs-2*np.sqrt(xs), 0.3), xs+2*np.sqrt(xs), color='0.85', zorder=0)
axs[1].set_xscale('log'); axs[1].set_yscale('log'); axs[1].set_xlim(lim); axs[1].set_ylim(0.3, lim[1])
axs[1].set_xlabel('Expected copies in column'); axs[1].set_ylabel('Placed copies'); axs[1].set_title('B  Packing fidelity (column)', loc='left', fontsize=10)
# C: structural source by mass in the column
srcs = {}
for (lid,_,_), tag in zip(pk.placed, pk.tags):
    reg = 'membranes' if tag in ('PM','ER_membrane_A','ER_membrane_B','NE_ONM','NE_INM','NPC') else ('nucleus' if tag in ('nucleoplasm','chromatin') else ('cytoplasm' if tag in ('cytosol','PM_peripheral','NE_peripheral','ER_lumen') else 'cell wall'))
    k = 'PDB' if lid in exp_ids else 'AlphaFold'
    srcs[(reg,k)] = srcs.get((reg,k),0) + libmass.get(lid,0)
sdf = pd.Series(srcs).unstack().fillna(0); sdf = sdf.div(sdf.sum(1), axis=0)
sdf[['PDB','AlphaFold']].plot.barh(stacked=True, ax=axs[2], color=[cb[2], cb[1]], width=0.7)
axs[2].set_xlabel('Fraction of placed macromolecular mass'); axs[2].set_ylabel('')
axs[2].legend(frameon=False, fontsize=8, ncol=2, loc='lower center', bbox_to_anchor=(0.5, 1.0))
axs[2].set_title('C  Structure source', loc='left', fontsize=10, pad=18)
sns.despine(fig); fig.tight_layout()
fig.savefig('/mnt/results/figures/fig3_composition_validation.png', dpi=300); fig.savefig('/mnt/results/figures/fig3_composition_validation.svg')
print(sdf.round(3))

           AlphaFold    PDB
cell wall      1.000  0.000
cytoplasm      0.457  0.543
membranes      0.603  0.397
nucleus        0.708  0.292


In [30]:
import export as ex, tarfile, shutil; importlib.reload(ex)
OUT = '/mnt/results'
os.makedirs(f'{OUT}/inventory', exist_ok=True); os.makedirs(f'{OUT}/model', exist_ok=True)
# inventory tables
df.assign(abundance_source='PaxDb v5 S. cerevisiae whole-organism integrated (ppm) rescaled to 4.2e7 proteins/cell (Ho et al. 2018)').to_csv(f'{OUT}/inventory/proteins_per_cell.csv', index=False)
comp.to_csv(f'{OUT}/inventory/compartments.csv', index=False)
isrc = ing.copy()
isrc['structure'] = ['PDB ' + {'ribosome_80S':'4V88','proteasome_26S':'4CR2','FAS':'2UV8','CCT':'4V94','PolII':'1WCM','PolI':'4C2M','PolIII':'5FJ8','NPC':'7N9F','Pma1_hexamer':'7VH5','V-ATPase':'6O7V','nucleosome':'1ID3'}[i] if k=='complex' else ('PDB ' + oligo[i+'_oligo'][1] + f' (homo-{oligo[i+"_oligo"][2]}-mer)' if i+'_oligo' in oligo else ('AlphaFold DB v6' if os.path.exists(f'/workspace/yc/lib/{i}.npz') else 'none')) for i,k in zip(isrc.ing_id, isrc.kind)]
isrc['packed_in_column'] = isrc.ing_id.isin(set(cexp.ing_id))
isrc.to_csv(f'{OUT}/inventory/ingredients.csv', index=False)
pd.DataFrame([('tRNA', 3.3e6, 'Waldron & Lacroute 1975 via Shah et al. 2013', 'PDB 1EHZ'), ('mRNA', 3.0e4, 'range 15,000-60,000 (Chu & von der Haar 2012); midpoint-type choice', 'schematic thread'),
              ('nucleosome', 73152, 'genome 12.07 Mb / assumed NRL 165 bp', 'PDB 1ID3'), ('NPC', 117, 'Winey et al. 1997 (mean per nucleus)', 'PDB 7N9F'),
              ('lipids (column)', V['lipids_in_column'], 'Ejsing et al. 2009 class composition; 0.65 nm2/lipid', 'bead model'),
              ('beta-glucan (column, glucose units)', bud['b13_glucose']+bud['b16_glucose'], 'wall = 20% of dry mass (assumed); composition Orlean 2012', 'fibril bead model'),
              ('mannan (column, mannose units)', bud['mannose'], 'mannoprotein 37.5% of wall, 85% carbohydrate (assumed)', 'brush bead model')],
             columns=['component','copies_per_cell_or_column','source','representation']).to_csv(f'{OUT}/inventory/non_protein_components.csv', index=False)
# placements
plc = ex.placements_table(pk); plc.to_csv(f'{OUT}/model/column_placements.csv.gz', index=False)
WO[WO.ok][['lib_id','src_comp','memb','r','u','v','z']].rename(columns={'lib_id':'ingredient','src_comp':'compartment','memb':'membrane','r':'radius_nm','u':'u_nm','v':'v_nm','z':'z_nm'}).to_csv(f'{OUT}/model/wholecell_slab_objects.csv.gz', index=False)
# recipe
wc_summary = {'slab_thickness_nm': wc.SLAB, 'objects_placed': int(WO.ok.sum()), 'objects_requested': int(len(WO)), 'n_npc_in_section': len(shp['npc']),
              'section_area_fraction': {wc.LABEL_NAMES[l]: float(ar[l]/inside) for l in range(3,15)}, 'note': 'organelle arrangement illustrative; volumes data-driven; one sphere per macromolecule'}
rec = ex.recipe(ing, cexp, clog, clib, oligo, exp_ids, geo, col.LAY, col.regions(), wc_summary)
json.dump(rec, open(f'{OUT}/model/yeast_column_recipe.json','w'), indent=1, default=lambda o: o.tolist() if hasattr(o,'tolist') else str(o))
# structure library (ingredients actually used)
used = sorted(set(p[0] for p in pk.placed))
with tarfile.open('/workspace/yc/structure_library.tar.gz', 'w:gz') as tf:
    for l in used: tf.add(f'/workspace/yc/lib/{l}.npz', arcname=f'structure_library/{l}.npz')
shutil.copy('/workspace/yc/structure_library.tar.gz', f'{OUT}/model/structure_library.tar.gz')
json.dump({k: float(v) if isinstance(v,(np.floating,float)) else v for k,v in V.items()}, open(f'{OUT}/model/validation_metrics.json','w'), indent=1, default=int)
print(len(used), 'library entries;', os.path.getsize(f'{OUT}/model/structure_library.tar.gz')/1e6, 'MB')

2593 library entries; 16.788645 MB


In [33]:
importlib.reload(ex)

<module 'export' from '/workspace/yc/export.py'>

In [34]:
# sub-slab: x in [0,100], y in [100,130] nm, full depth (wall -> nucleoplasm); whole macromolecules whose centres fall inside
sel_inst = np.array([(0 <= t[0] < 100) and (100 <= t[1] < 130) for (_,_,t) in pk.placed])
sel_inst[0] = True  # keep the NPC
m = sel_inst[pinst]
P, I, K = ppos[m], pinst[m], pkind[m]
# sequence index within each instance
order = np.argsort(I, kind='stable'); seq = np.empty(len(I), int); _, start, cnt = np.unique(I[order], return_index=True, return_counts=True)
seq[order] = np.arange(len(I)) - np.repeat(start, cnt) + 1
asym = np.char.add('P', I.astype(str))
items = [(P[K==0], 'UNK', 'CA', 'C', asym[K==0], seq[K==0]), (P[K==1], 'N', 'P', 'P', asym[K==1], seq[K==1])]
inbox = lambda X: (X[:,0] >= 0) & (X[:,0] < 100) & (X[:,1] >= 100) & (X[:,1] < 130)
lm = inbox(lip['pos']); head = lip['rad'] > 0.5
for sel, cid, at, el in [(lm & head, 'LPH', 'P', 'P'), (lm & ~head, 'LPT', 'C1', 'C')]:
    items.append((lip['pos'][sel], cid, at, el, np.char.add('L', lip['leaf'][sel].astype(str)), lip['inst'][sel]))
gm = inbox(glc); items.append((glc[gm], 'BGC', 'C1', 'C', np.char.add('G', glc_fid[gm].astype(str)), np.arange(gm.sum())+1))
mm = inbox(man); items.append((man[mm], 'MAN', 'C1', 'C', np.char.add('M', man_owner[mm].astype(str)), np.arange(mm.sum())+1))
lk = inbox(linker); items.append((linker[lk], 'DN', 'P', 'P', np.full(lk.sum(), 'D0'), np.arange(lk.sum())+1))
mr = np.vstack(mrna); mid = np.concatenate([np.full(len(c), i) for i, c in enumerate(mrna)]); rm_ = inbox(mr)
items.append((mr[rm_], 'N', 'P', 'P', np.char.add('R', mid[rm_].astype(str)), np.arange(rm_.sum())+1))
nat = ex.write_mmcif('/mnt/results/model/column_subslab_100nm.cif.gz', items)
print('atoms', nat, 'macromolecules', sel_inst.sum(), 'size MB', os.path.getsize('/mnt/results/model/column_subslab_100nm.cif.gz')/1e6)
import gemmi; st = gemmi.read_structure('/mnt/results/model/column_subslab_100nm.cif.gz'); print('gemmi read OK:', st[0].count_atom_sites(), 'atoms,', len(st[0]), 'chains')

atoms 1050197 macromolecules 1416 size MB 18.39499
gemmi read OK: 1050197 atoms, 1611 chains


In [35]:
cnt = pd.Series([p[0] for p in pk.placed]).value_counts()
print({k: int(cnt.get(k,0)) for k in ['ribosome_80S','tRNA','nucleosome','proteasome_26S','FAS','CCT','PolI','PolII','PolIII','Pma1_hexamer','NPC']})
print(pd.Series(pk.tags).value_counts().to_dict())
print('distinct ingredients in column', len(cnt), '| mRNA chains', len(mrna), '| glucan fibrils', glc_fid.max()+1, '| mannan beads', len(man))
print('whole-cell ribosomes', int(((WO.lib_id=='ribosome_80S')&WO.ok).sum()), 'nucleosomes', int(((WO.lib_id=='nucleosome')&WO.ok).sum()), 'tRNA', int(((WO.lib_id=='tRNA')&WO.ok).sum()))
os.makedirs('/mnt/results/figures/renders_full_resolution', exist_ok=True)
for f in ['column_full.png','wholecell.png','wholecell_zoom.png','crop_wall.png','crop_nucleus.png']:
    shutil.copy(f'/workspace/yc/fig/{f}', f'/mnt/results/figures/renders_full_resolution/{f}')
with tarfile.open('/workspace/yc/code.tar.gz','w:gz') as tf:
    for f in ['inventory.py','ingredients.py','structlib.py','build_structs.py','packer.py','column.py','assemble.py','render.py','style.py','wholecell.py','figs.py','export.py','fetch_uniprot.py']:
        tf.add(f'/workspace/yc/{f}', arcname=f'yeast_cell_model_code/{f}')
shutil.copy('/workspace/yc/code.tar.gz', '/mnt/results/model/code.tar.gz')
for root, _, fs in os.walk('/mnt/results'):
    if 'execution_trace' in root: continue
    for f in fs: print(os.path.join(root, f).replace('/mnt/results/',''), round(os.path.getsize(os.path.join(root,f))/1e6,2), 'MB')

{'ribosome_80S': 173, 'tRNA': 3481, 'nucleosome': 256, 'proteasome_26S': 17, 'FAS': 9, 'CCT': 5, 'PolI': 0, 'PolII': 19, 'PolIII': 8, 'Pma1_hexamer': 50, 'NPC': 1}
{'cytosol': 19826, 'nucleoplasm': 4828, 'NE_peripheral': 502, 'cell_wall': 436, 'ER_membrane_A': 369, 'ER_membrane_B': 344, 'PM_peripheral': 340, 'ER_lumen': 312, 'PM': 282, 'chromatin': 256, 'NE_INM': 66, 'NE_ONM': 59, 'NPC': 1}
distinct ingredients in column 2593 | mRNA chains 31 | glucan fibrils 568 | mannan beads 438093
whole-cell ribosomes 1343 nucleosomes 1628 tRNA 27217


.keep 0.0 MB
figures/fig1_column_goodsell.png 13.68 MB
figures/fig1_column_goodsell.svg 18.22 MB
figures/fig2_wholecell_goodsell.png 10.6 MB
figures/fig2_wholecell_goodsell.svg 14.01 MB
figures/fig3_composition_validation.png 0.25 MB
figures/fig3_composition_validation.svg 0.46 MB
figures/renders_full_resolution/column_full.png 10.6 MB
figures/renders_full_resolution/crop_nucleus.png 3.26 MB
figures/renders_full_resolution/crop_wall.png 3.73 MB
figures/renders_full_resolution/wholecell.png 26.91 MB
figures/renders_full_resolution/wholecell_zoom.png 4.85 MB
inventory/compartments.csv 0.0 MB
inventory/ingredients.csv 1.09 MB
inventory/non_protein_components.csv 0.0 MB
inventory/proteins_per_cell.csv 2.76 MB
model/code.tar.gz 0.03 MB
model/column_placements.csv.gz 0.8 MB
model/column_subslab_100nm.cif.gz 18.39 MB
model/structure_library.tar.gz 16.79 MB
model/validation_metrics.json 0.0 MB
model/wholecell_slab_objects.csv.gz 8.04 MB
model/yeast_column_recipe.json 1.11 MB


# Part 2: 3D whole-cell model + WebGPU viewer
Inventory v2 → 3D SDF geometry → tiled 3D packing → validation → web data encoding.

In [39]:
import sys, os, json, time, glob, importlib; sys.path.insert(0, '/workspace/yc')
import numpy as np, pandas as pd
import inventory as inv, ingredients as ingm, build_structs as bs, structlib as sl, extend_complexes as xc
importlib.reload(xc)
df, geo = inv.build(); comp = inv.per_compartment(df, geo)
ing, consumed = ingm.build_ingredients(df)
# homo-oligomers from the saved library
oligo_map = {}
for f in glob.glob('/workspace/yc/lib/*_oligo.npz'):
    g = os.path.basename(f)[:-10]; oligo_map[g] = (g + '_oligo', int(ing.set_index('ing_id').loc[g, 'oligomer_n']))
modeled = set(g for gs in ing[ing.kind=='complex'].genes for g in gs.split(';')) | set(oligo_map)
t0 = time.time()
chosen, tried = xc.run(df, modeled)
ing2 = xc.apply(ing, df, chosen)
# excess free ribosomal-protein pool is not packed (as in the column model)
ing2 = ing2[~((ing2.kind=='protein') & ing2.ing_id.str.match(r'^RP[LS]\d'))].reset_index(drop=True)
print(f'{len(chosen)} complexes added in {time.time()-t0:.0f}s; ingredients {len(ing2)}; copies {ing2.copies_per_cell.sum()/1e6:.2f}M')
print(tried.outcome.value_counts().to_string())
os.makedirs('/mnt/results/inventory', exist_ok=True)
ing2.to_csv('/mnt/results/inventory/ingredients_v2.csv', index=False)
tried.to_csv('/mnt/results/inventory/complexes_tried_v2.csv', index=False)
pd.DataFrame(chosen).drop(columns=['sub']).assign(subunits=[json.dumps(c['sub']) for c in chosen])

CPX-729 Box C/D snoRNA-guided RNP methyltransfer 9N74 cov=0.81 res=None copies=25440 sub={'NOP1': 2, 'SNU13': 2, 'NOP56': 1, 'NOP58': 1}


CPX-26769 RAN-RANBP1-CRM1 nuclear export complex   3M1I cov=0.87 res=None copies=26092 sub={'CRM1': 1, 'GSP1': 1, 'YRB1': 1}


CPX-26749 Trehalose hydrolysis complex, NTH1-BMH1  5N6N cov=0.88 res=None copies=22365 sub={'BMH1': 2, 'NTH1': 1}


CPX-3034 Acetolactate synthase complex            6U9D cov=0.87 res=None copies=2583 sub={'ILV2': 8, 'ILV6': 8}


CPX-1283 HTM1-PDI1 exomannosidase complex         8ZPW cov=0.84 res=None copies=14832 sub={'PDI1': 1, 'MNL1': 1}


CPX-1409 UTP-A complex                            9G33 cov=0.81 res=None copies=3131 sub={'UTP9': 1, 'UTP10': 1, 'UTP8': 1, 'NAN1': 1, 'UTP5': 1, 'UTP15': 1, 'UTP4': 1}


CPX-1424 Tubulin alpha-beta heterodimeric complex 5W3J cov=0.95 res=None copies=18008 sub={'TUB2': 1, 'TUB1': 1}
7 complexes added in 53s; ingredients 5637; copies 26.92M
outcome
no PDB assembly with >=80% subunit-mass coverage    31
added                                                7
overlaps chosen complex                              2


,ing_id,cpx,label,copies,mass_da,pdb,resolution,coverage,n_beads,subunits
0,CPX_729_Box_C_D_snoRNA_guided_RNP_methyl,CPX-729,Box C/D snoRNA-guided RNP methyltransferase co...,25440.107676,209888,9N74,None,0.809433,1550,"{""NOP1"": 2, ""SNU13"": 2, ""NOP56"": 1, ""NOP58"": 1}"
1,CPX_26769_RAN_RANBP1_CRM1_nuclear_export_c,CPX-26769,RAN-RANBP1-CRM1 nuclear export complex,26092.418129,171867,3M1I,None,0.866600,1354,"{""CRM1"": 1, ""GSP1"": 1, ""YRB1"": 1}"
2,CPX_26749_Trehalose_hydrolysis_complex_NTH,CPX-26749,"Trehalose hydrolysis complex, NTH1-BMH1 variant",22364.624293,146061,5N6N,None,0.881176,1153,"{""BMH1"": 2, ""NTH1"": 1}"
3,CPX_3034_Acetolactate_synthase_complex,CPX-3034,Acetolactate synthase complex,2582.507778,871392,6U9D,None,0.870515,6820,"{""ILV2"": 8, ""ILV6"": 8}"
4,CPX_1283_HTM1_PDI1_exomannosidase_complex,CPX-1283,HTM1-PDI1 exomannosidase complex,14832.042600,149473,8ZPW,None,0.842627,1145,"{""PDI1"": 1, ""MNL1"": 1}"
5,CPX_1409_UTP_A_complex,CPX-1409,UTP-A complex,3131.090175,664270,9G33,None,0.812245,4905,"{""UTP9"": 1, ""UTP10"": 1, ""UTP8"": 1, ""NAN1"": 1, ..."
6,CPX_1424_Tubulin_alpha_beta_heterodimeric,CPX-1424,"Tubulin alpha-beta heterodimeric complex, TUB1...",18008.045955,100723,5W3J,None,0.946854,867,"{""TUB2"": 1, ""TUB1"": 1}"


## Phase 2 / Step 1b: 3D whole-cell geometry (analytic SDF primitives)

In [41]:
import cell3d as c3; importlib.reload(c3)
cell = c3.build(geo, seed=11)
os.makedirs('/workspace/yc3d', exist_ok=True); cell.to_json('/workspace/yc3d/geometry.json')
print('prims', len(cell.PT), 'pores', len(cell.pores))
# Monte-Carlo volume check (all prims, 4M points in the cell sphere)
rng = np.random.default_rng(0); N = 4_000_000
R0 = cell.G[0]
P = rng.uniform(-R0, R0, (N, 3)); P = P[np.linalg.norm(P, axis=1) < R0]
lab = c3.label_points(P, cell.G, cell.pores, np.arange(len(cell.PT)), cell.PT, cell.PQ, cell.PC)
Vc = 4/3*np.pi*(R0/1000)**3
vol = pd.Series(np.bincount(lab, minlength=15) / len(P) * Vc, index=c3.LABEL_NAMES)
target = {'vacuole': geo['volumes_um3']['vacuole'], 'mito_IMS+matrix': geo['volumes_um3']['mitochondrion'],
          'lipid_droplet': geo['volumes_um3']['lipid_droplet'], 'golgi_endosome': geo['volumes_um3']['golgi_endosome'],
          'peroxisome': geo['volumes_um3']['peroxisome'], 'nucleolus': geo['volumes_um3']['nucleolus'],
          'nucleoplasm': geo['volumes_um3']['nucleoplasm'], 'wall': geo['volumes_um3']['cell_wall'], 'cytosol': geo['volumes_um3']['cytosol']}
print(vol.round(3).to_string())
print('mito total', round(vol.mito_IMS + vol.mito_matrix, 3), 'wall', round(vol.wall_mannan + vol.wall_glucan, 3))
print('targets', {k: round(v, 3) for k, v in target.items()})

mito: 30 segments, length 10339 nm, est. volume 0.858 um3 (target 0.840)
cristae: 218
LD r=216  perox r=171  endosome r=206  golgi cisternae 14
cER: 39 patches, PM coverage 0.482
prims 362 pores 117


exterior           0.000
wall_mannan        2.007
wall_glucan        4.095
cytosol           27.843
cER_lumen          0.504
perinuclear        0.296
nucleoplasm        2.005
nucleolus          0.590
vacuole            2.927
mito_IMS           0.209
mito_matrix        0.646
lipid_droplet      0.418
golgi_endosome     0.336
peroxisome         0.083
pore_channel       0.041
mito total 0.855 wall 6.102
targets {'vacuole': 2.94, 'mito_IMS+matrix': 0.84, 'lipid_droplet': 0.42, 'golgi_endosome': 0.336, 'peroxisome': 0.084, 'nucleolus': 0.588, 'nucleoplasm': 2.352, 'wall': 6.105, 'cytosol': 27.431}


In [42]:
import tiles3d as t3; importlib.reload(t3)
tiles = t3.tile_list(cell)
t0 = time.time()
gres = t3.run_geometry('/workspace/yc3d/geometry.json', tiles, nproc=16)
print(f'{len(gres)} tiles in {time.time()-t0:.0f}s')
tkeys = [r[0] for r in gres]
measure = pd.DataFrame(np.array([r[1] for r in gres]), columns=t3.REGION_NAMES, index=[t3.tile_name(t) for t in tkeys])
labvol = pd.DataFrame(np.array([r[2] for r in gres]), columns=c3.LABEL_NAMES, index=measure.index)
measure.to_parquet('/workspace/yc3d/tile_measure.parquet'); labvol.to_parquet('/workspace/yc3d/tile_labvol.parquet')
tot = measure.sum()
print('volumes (um3):'); print((tot[t3.VOL_REGIONS] / 1e9).round(3).to_string())
print('areas (um2):'); print((tot[['mem_' + m for m in t3.MEM_REGIONS]] / 1e6).round(2).to_string())
print('faces total', sum(r[3] for r in gres))

6416 tiles in 27s
volumes (um3):
cytosol           27.813
nucleoplasm        2.017
nucleolus          0.587
ER_lumen           0.799
vacuole            2.940
mito_matrix        0.650
mito_IMS           0.208
golgi_endosome     0.336
peroxisome         0.084
cell_wall          3.409
PM_peripheral      1.146
NE_peripheral      0.192
ER_peripheral      1.390
LD_peripheral      0.074
areas (um2):
mem_PM         52.62
mem_ER         51.82
mem_NE         18.29
mem_VAC         9.92
mem_MITO_OM    12.28
mem_MITO_IM    27.97
mem_GOLGI       8.92
mem_PEROX       1.47
faces total 8906828


## Phase 2 / Step 1c: 3D packing (expected counts, NPC + chromatin pre-pass, tiles)

In [43]:
import pack3d as p3, wc3d
for m in (c3, t3, p3, wc3d): importlib.reload(m)
cell = c3.Cell.load('/workspace/yc3d/geometry.json')
exp3 = wc3d.expected_totals(ing2, oligo_map)
print(exp3.groupby('region').expected.sum().round(0).to_string())
lib_ids = sorted(set(exp3[exp3.region.isin(t3.REGION_NAMES)].lib_id) | {'NPC', 'nucleosome'})
lp = wc3d.build_libpack(lib_ids)
lib_index = {k: i for i, k in enumerate(lp['ids'])}
halo = float(max(lp['BR'][i] for k, i in lib_index.items() if k != 'NPC')) + 1.0
print(f'{len(lib_ids)} library types, {len(lp["STR"])} sphere-tree spheres, halo {halo:.1f} nm')
counts = wc3d.distribute(exp3, measure, lib_index)
print(f'{len(counts)} (tile,lib,region) entries, {counts.n.sum()/1e6:.2f}M instances to place')
rng = np.random.default_rng(9)
w = measure['cytosol'].values; mrna_counts = rng.multinomial(int(wc3d.N_MRNA), w / w.sum())
n_nuc = int(round(ing2.set_index('ing_id').loc['nucleosome', 'copies_per_cell']))
pre = wc3d.prepass(cell, lp, n_nuc)
np.savez('/workspace/yc3d/prepass.npz', **pre)

region
ER_lumen                               272724.0
ER_peripheral                            5070.0
LD_peripheral                           83614.0
NE_peripheral                           86440.0
PM_peripheral                          289137.0
cell_wall                              388863.0
cytosol                              18743619.0
dropped_TM_in_soluble_compartment       63238.0
golgi_endosome                         318130.0
mem_ER                                 581988.0
mem_GOLGI                              130373.0
mem_MITO_IM                            313386.0
mem_MITO_OM                            104462.0
mem_NE                                  20471.0
mem_PEROX                               26673.0
mem_PM                                 242405.0
mem_VAC                                110659.0
mito_IMS                               234510.0
mito_matrix                           2110587.0
no_structure                             6888.0
nucleolus                        

5483 library types, 82562 sphere-tree spheres, halo 36.8 nm


4124410 (tile,lib,region) entries, 26.28M instances to place


chromatin: 73150/73150 nucleosomes, 56 fragments, 2s


In [44]:
import shutil; shutil.rmtree(wc3d.RAW, ignore_errors=True)
# measures were collected in imap order: re-index to the canonical tile list order before distributing
measure = measure.loc[[t3.tile_name(t) for t in tiles]]; labvol = labvol.loc[measure.index]
measure.to_parquet('/workspace/yc3d/tile_measure.parquet'); labvol.to_parquet('/workspace/yc3d/tile_labvol.parquet')
counts = wc3d.distribute(exp3, measure, lib_index)
rng = np.random.default_rng(9)
w = measure['cytosol'].values; mrna_counts = rng.multinomial(int(wc3d.N_MRNA), w / w.sum())
counts.to_parquet('/workspace/yc3d/tile_counts.parquet'); np.save('/workspace/yc3d/mrna_counts.npy', mrna_counts)
wc3d.setup_shared(cell, lp, counts, tiles, pre, halo)
phase = np.array([wc3d.phase_of(t) for t in tiles])
rng = np.random.default_rng(1)
pilot = rng.choice(np.where(phase == 0)[0], 48, replace=False)
t0 = time.time()
pres = wc3d.run_phase(list(pilot), mrna_counts, nproc=16)
wall = time.time() - t0
pr = pd.DataFrame(pres, columns=['ti', 'requested', 'placed', 'overflow', 'sec', 'n_mrna'])
pr['frac'] = pr.placed / pr.requested.clip(lower=1)
print(f'pilot wall {wall:.0f}s; per-tile sec median {pr.sec.median():.1f}, max {pr.sec.max():.1f}; overflow {pr.overflow.sum()}')
print(f'placed {pr.placed.sum()}/{pr.requested.sum()} ({pr.placed.sum()/pr.requested.sum():.3f}); inst/s/proc {pr.placed.sum()/pr.sec.sum():.0f}')
tot_req = counts.n.sum(); est = tot_req / (pr.placed.sum() / pr.sec.sum()) / 16 / 60
print(f'extrapolated full run ~{est:.0f} min on 16 processes')
pr.sort_values('frac').head(8)

pilot wall 4s; per-tile sec median 0.1, max 1.5; overflow 0
placed 182174/188977 (0.964); inst/s/proc 36175
extrapolated full run ~1 min on 16 processes


,ti,requested,placed,overflow,sec,n_mrna,frac
43,4993,0,0,False,0.007133,0,0.000000
40,6211,0,0,False,0.000943,0,0.000000
41,5715,0,0,False,0.001251,0,0.000000
47,5030,18574,12384,False,1.489997,4,0.666738
26,2735,6338,5886,False,0.283742,6,0.928684
34,2607,2649,2613,False,0.045268,1,0.986410
11,533,8325,8230,False,0.299176,0,0.988589
44,3419,4462,4448,False,0.076684,7,0.996862


In [45]:
import shutil, tarfile, subprocess
shutil.rmtree(wc3d.RAW, ignore_errors=True); os.makedirs(wc3d.RAW)
CK = '/mnt/shared-workspace/yc3d'; os.makedirs(CK, exist_ok=True)
allres = []
T0 = time.time()
for ph in range(8):
    ids = list(np.where(phase == ph)[0])
    t0 = time.time()
    r = wc3d.run_phase(ids, mrna_counts, nproc=16)
    allres += r
    # checkpoint this phase (tar on local disk, then copy) and verify
    names = [t3.tile_name(tiles[i]) + '.npz' for i in ids]
    tf = f'/workspace/yc3d/phase{ph}.tar'
    with tarfile.open(tf, 'w') as tar:
        for n in names: tar.add(f'{wc3d.RAW}/{n}', arcname=n)
    shutil.copy(tf, f'{CK}/phase{ph}.tar')
    print(f'phase {ph}: {len(ids)} tiles, {time.time()-t0:.0f}s, placed {sum(x[2] for x in r)}/{sum(x[1] for x in r)}, overflow {sum(x[3] for x in r)}')
print(f'total {time.time()-T0:.0f}s')
res3 = pd.DataFrame(allres, columns=['ti', 'requested', 'placed', 'overflow', 'sec', 'n_mrna'])
res3.to_parquet('/workspace/yc3d/pack_summary.parquet')
print(f'placed {res3.placed.sum()/1e6:.2f}M / {res3.requested.sum()/1e6:.2f}M = {res3.placed.sum()/res3.requested.sum():.4f}')

phase 0: 802 tiles, 12s, placed 3160743/3294359, overflow 0


phase 1: 802 tiles, 13s, placed 3147056/3292404, overflow 0


phase 2: 802 tiles, 13s, placed 3122481/3246390, overflow 0


phase 3: 802 tiles, 14s, placed 3113354/3246757, overflow 0


phase 4: 802 tiles, 17s, placed 3166855/3315589, overflow 0


phase 5: 802 tiles, 18s, placed 3155276/3315542, overflow 0


phase 6: 802 tiles, 18s, placed 3137859/3274676, overflow 0


phase 7: 802 tiles, 18s, placed 3136932/3295220, overflow 0
total 123s
placed 25.14M / 26.28M = 0.9566


## Phase 2 / Step 1d: validation of the 3D whole cell

In [46]:
import val3d as v3; importlib.reload(v3)
TY, PO, QU, TI, JOB, MRP, MRT = v3.load_instances(tiles, wc3d.RAW)
# region per instance (instances are written in job order)
REG = np.concatenate([np.repeat(g.region.values, g.placed.values) for _, g in JOB.groupby('tile', sort=True)]).astype(np.int16)
assert len(REG) == len(TY)
# append global pre-pass instances
TYa = np.concatenate([TY, np.full(len(pre['npc_pos']), pre['li_npc'], np.int32), np.full(len(pre['nuc_pos']), pre['li_nuc'], np.int32)])
POa = np.concatenate([PO, pre['npc_pos'], pre['nuc_pos']]); QUa = np.concatenate([QU, pre['npc_q'], pre['nuc_q']])
TIa = np.concatenate([TI, np.full(len(pre['npc_pos']) + len(pre['nuc_pos']), -1, np.int32)])
boff, BB, BK, LMASS, LSRC = v3.lib_beads(lp['ids'])
print(f'instances {len(TYa)/1e6:.2f}M; residue/nt beads in cell {np.diff(boff)[TYa].sum()/1e9:.2f}G; mRNA beads {len(MRP)/1e6:.1f}M')

# ---- placed vs requested vs expected, per region
reqreg = JOB.groupby('region')[['n', 'placed']].sum()
expreg = exp3[exp3.region.isin(t3.REGION_NAMES)].groupby('region').expected.sum()
reg_tab = pd.DataFrame({'expected': expreg.reindex(t3.REGION_NAMES).values, 'requested': reqreg.n.reindex(range(t3.NREG)).values,
                        'placed': reqreg.placed.reindex(range(t3.NREG)).values}, index=t3.REGION_NAMES).fillna(0)
reg_tab['frac_placed'] = reg_tab.placed / reg_tab.requested.clip(lower=1)
# Poisson z of requested cell totals vs expected per (lib, region) (checks the stochastic rounding/multinomial split)
lr = JOB.groupby(['lib', 'region']).n.sum().reset_index()
lr['lib_id'] = lp['ids'][lr.lib]; lr['region_name'] = np.array(t3.REGION_NAMES)[lr.region]
e2 = exp3.groupby(['lib_id', 'region']).expected.sum()
lr['expected'] = [e2.get((a, b), 0) for a, b in zip(lr.lib_id, lr.region_name)]
zz = (lr.n - lr.expected) / np.sqrt(lr.expected.clip(lower=1))
print(reg_tab.round(3).to_string())
print(f'per (ingredient,region) requested-vs-expected |z|: median {zz.abs().median():.2f}, frac |z|<2 {np.mean(zz.abs()<2):.4f}')

instances 25.21M; residue/nt beads in cell 14.04G; mRNA beads 28.8M


                    expected  requested    placed  frac_placed
cytosol         1.874362e+07   18743624  18741626        1.000
nucleoplasm     1.351980e+06    1352011   1351894        1.000
nucleolus       6.074612e+05     607466    586273        0.965
ER_lumen        2.727240e+05     272721    271023        0.994
vacuole         1.566067e+05     156608    156607        1.000
mito_matrix     2.110587e+06    2110584   1157052        0.548
mito_IMS        2.345096e+05     234509     99403        0.424
golgi_endosome  3.181298e+05     318129    311117        0.978
peroxisome      1.017464e+05     101749    101698        0.999
cell_wall       3.888630e+05     388856    388856        1.000
PM_peripheral   2.891371e+05     289133    289014        1.000
NE_peripheral   8.644001e+04      86437     86380        0.999
ER_peripheral   5.070056e+03       5070      4461        0.880
LD_peripheral   8.361359e+04      83613     79911        0.956
mem_PM          2.424050e+05     242409    242331      

In [47]:
from scipy.spatial import cKDTree
t0 = time.time()
ctree = cKDTree(POa)
mtree = cKDTree(MRP)
BRa = lp['BR']
rng = np.random.default_rng(42)
# 64 cubes of 50 nm: half centred on tile corners (cross-tile pairs), half random inside the protoplast
corners = [np.array(tiles[i], float) * t3.TILE for i in rng.choice(len(tiles), 400)]
corners = [c for c in corners if np.linalg.norm(c) < cell.G[2] - 60][:32]
randc = []
while len(randc) < 32:
    p = rng.uniform(-2000, 2000, 3)
    if np.linalg.norm(p) < cell.G[2] - 40: randc.append(p)
half = 25.0
n_beads = n_clash = n_clash_x = n_close1 = 0
for cc in corners + randc:
    cand = np.array(ctree.query_ball_point(cc, half * np.sqrt(3) + 66.0), np.int64)
    if len(cand) == 0: continue
    cand = cand[np.linalg.norm(POa[cand] - cc, axis=1) < half * np.sqrt(3) + BRa[TYa[cand]] + 0.5]
    nb_ = np.diff(boff)[TYa[cand]].sum()
    pts = np.empty((nb_, 3)); inst = np.empty(nb_, np.int64)
    v3.beads_of(cand, TYa, POa.astype(np.float64), QUa.astype(np.float64), boff, BB, pts, inst)
    k = np.all(np.abs(pts - cc) < half + 0.5, axis=1); pts, inst = pts[k], inst[k]
    mi = np.array(mtree.query_ball_point(cc, half * np.sqrt(3) + 1), np.int64)
    if len(mi):
        mp_ = MRP[mi].astype(np.float64); kk = np.all(np.abs(mp_ - cc) < half + 0.5, axis=1)
        pts = np.vstack([pts, mp_[kk]]); inst = np.concatenate([inst, -2 - mi[kk]])
    tr = cKDTree(pts)
    core = np.all(np.abs(pts - cc) < half, axis=1)
    pairs = tr.query_pairs(0.3, output_type='ndarray')
    a, b = inst[pairs[:, 0]], inst[pairs[:, 1]]
    # different molecules; consecutive mRNA beads of one thread are > 0.3 nm apart so mRNA ids can be per-bead
    diff = a != b
    bad = np.zeros(len(pts), bool); bad[pairs[diff, 0]] = True; bad[pairs[diff, 1]] = True
    def tile_of(x): return np.where(x >= 0, TIa[np.clip(x, 0, None)], -9)
    xt = diff & (tile_of(a) != tile_of(b)) & (a >= 0) & (b >= 0)
    badx = np.zeros(len(pts), bool); badx[pairs[xt, 0]] = True; badx[pairs[xt, 1]] = True
    n_beads += core.sum(); n_clash += (bad & core).sum(); n_clash_x += (badx & core).sum()
clash = dict(beads_sampled=int(n_beads), beads_clashing_lt_0p3nm=int(n_clash), clash_rate=n_clash / n_beads,
             cross_tile_clashing_beads=int(n_clash_x), cubes=len(corners) + len(randc))
print(clash, f'{time.time()-t0:.0f}s')

{'beads_sampled': 2833270, 'beads_clashing_lt_0p3nm': 13, 'clash_rate': np.float64(4.5883378569638615e-06), 'cross_tile_clashing_beads': 0, 'cubes': 64} 70s


In [48]:
allp = np.arange(len(cell.PT))
rng = np.random.default_rng(7)
n_inst = len(TY)
PO64, QU64 = PO.astype(np.float64), QU.astype(np.float64)
# ---- containment of residue beads (volume-region instances)
volidx = np.where(REG < t3.MEM_REGION0)[0]; s = rng.choice(volidx, 30000, replace=False)
nb_ = np.diff(boff)[TY[s]].sum(); pts = np.empty((nb_, 3)); inst = np.empty(nb_, np.int64)
v3.beads_of(s, TY, PO64, QU64, boff, BB, pts, inst)
labb = c3.label_points(pts, cell.G, cell.pores, allp, cell.PT, cell.PQ, cell.PC)
soft = {c3.NUCP: {c3.NUCL}, c3.NUCL: {c3.NUCP}, c3.MANNAN: {c3.GLUCAN}, c3.GLUCAN: {c3.MANNAN}}
mask_b = t3.REG_MASK[REG[inst]]
ok = ((mask_b >> labb) & 1).astype(bool)
# allow soft (membrane-less) boundaries
for l, alts in soft.items():
    for a in alts:
        ok |= (((mask_b >> l) & 1).astype(bool)) & (labb == a)
bad_inst = np.unique(inst[~ok])
contain = dict(volume_instances_sampled=30000, beads=int(nb_), frac_beads_outside_region=float((~ok).mean()),
               frac_instances_any_bead_outside=float(len(bad_inst) / 30000))
# ---- membrane orientation + TM-bead depth
memidx = np.where(REG >= t3.MEM_REGION0)[0]; s = rng.choice(memidx, 10000, replace=False)
ang = []; out = np.zeros(c3.NMEM); n_ = np.empty(3)
for i in s:
    m = REG[i] - t3.MEM_REGION0; M = np.empty((3, 3)); p3.quat_mat(QU64[i], M)
    p = PO64[i].copy(); ok_ = p3.project_to_membrane(p, m, n_, cell.G, cell.pores, allp, cell.PT, cell.PQ, cell.PC, out)
    ang.append(np.degrees(np.arccos(np.clip(M[:, 2] @ n_, -1, 1))))
ang = np.array(ang)
orient = dict(membrane_instances_sampled=10000, median_angle_deg=float(np.median(ang)), frac_angle_lt_1deg=float((ang < 1).mean()))
# ---- density per compartment (mg/mL) and PDB vs AF mass
mass_inst = LMASS[TY]
reg2comp = {'cytosol': 'cytosol', 'PM_peripheral': 'cytosol', 'NE_peripheral': 'cytosol', 'ER_peripheral': 'cytosol', 'LD_peripheral': 'cytosol',
            'nucleoplasm': 'nucleoplasm', 'nucleolus': 'nucleolus', 'ER_lumen': 'ER_lumen', 'vacuole': 'vacuole', 'mito_matrix': 'mito_matrix',
            'mito_IMS': 'mito_IMS', 'golgi_endosome': 'golgi_endosome', 'peroxisome': 'peroxisome', 'cell_wall': 'cell_wall'}
rn_ = np.array(t3.REGION_NAMES)[REG]
dfm = pd.DataFrame(dict(comp=pd.Series(rn_).map(reg2comp), mass=mass_inst))
cm = dfm.groupby('comp').mass.sum()
cm['nucleoplasm'] += len(pre['nuc_pos']) * LMASS[pre['li_nuc']]
lv = labvol.sum()
cvol = {'cytosol': lv.cytosol, 'nucleoplasm': lv.nucleoplasm, 'nucleolus': lv.nucleolus, 'ER_lumen': lv.cER_lumen + lv.perinuclear,
        'vacuole': lv.vacuole, 'mito_matrix': lv.mito_matrix, 'mito_IMS': lv.mito_IMS, 'golgi_endosome': lv.golgi_endosome,
        'peroxisome': lv.peroxisome, 'cell_wall': lv.wall_mannan + lv.wall_glucan}
DA_G = 1.66054e-24
mgml = {k: float(cm.get(k, 0) * DA_G / (v * 1e-21) * 1000) for k, v in cvol.items()}   # g/cm3 -> mg/mL
srcm = pd.Series(mass_inst).groupby(LSRC[TY]).sum(); srcm = (srcm / srcm.sum()).to_dict()
mito = REG.astype(int)
nonmito = ~reg_tab.index.isin(['mito_matrix', 'mito_IMS', 'mem_MITO_OM', 'mem_MITO_IM'])
frac_nonmito = float(reg_tab[nonmito].placed.sum() / reg_tab[nonmito].requested.sum())
print(contain); print(orient); print({k: round(v, 1) for k, v in mgml.items()}); print('mass by source', srcm); print('placed fraction outside mitochondria', round(frac_nonmito, 5))

{'volume_instances_sampled': 30000, 'beads': 16785810, 'frac_beads_outside_region': 7.810168231381149e-05, 'frac_instances_any_bead_outside': 0.0004}
{'membrane_instances_sampled': 10000, 'median_angle_deg': 0.0009608943743392357, 'frac_angle_lt_1deg': 1.0}
{'cytosol': 88.7, 'nucleoplasm': 63.2, 'nucleolus': 89.3, 'ER_lumen': 36.3, 'vacuole': 4.9, 'mito_matrix': 109.9, 'mito_IMS': 25.5, 'golgi_endosome': 65.3, 'peroxisome': 84.8, 'cell_wall': 4.5}
mass by source {'AF': 0.8126538166956596, 'PDB': 0.1873461833043404}
placed fraction outside mitochondria 0.99839


In [49]:
# ---- structure source per library type (PDB entry from the inventory; everything else AlphaFold DB v6)
i2 = ing2.set_index('ing_id')
pdbcol = i2.pdb.fillna('').astype(str).str.upper()
fixed = {'NPC': '7N9F', 'nucleosome': '1ID3', 'tRNA': '1EHZ'}
def src_of(lid):
    if lid in fixed: return 'PDB', fixed[lid]
    key = lid[:-6] if lid.endswith('_oligo') else lid
    p = pdbcol.get(key, '')
    if p and p not in ('NAN', 'NONE'): return 'PDB', p
    return 'AF', ''
SRC2 = np.array([src_of(str(i))[0] for i in lp['ids']]); PDBID = np.array([src_of(str(i))[1] for i in lp['ids']])
mass_all = np.concatenate([LMASS[TY], np.full(len(pre['npc_pos']), LMASS[pre['li_npc']]), np.full(len(pre['nuc_pos']), LMASS[pre['li_nuc']])])
src_all = SRC2[TYa]
srcm = pd.Series(mass_all).groupby(src_all).sum(); srcm = (srcm / srcm.sum()).round(4).to_dict()
srcn = pd.Series(src_all).value_counts(normalize=True).round(4).to_dict()
print('mass by source', srcm, '| instances by source', srcn, '| PDB types', int((SRC2 == 'PDB').sum()))
reg_tab['unplaced'] = reg_tab.requested - reg_tab.placed
val = dict(
    model='S. cerevisiae BY4741 haploid, YPD exponential; whole-cell 3D packing in 200 nm tiles (8 checkerboard phases)',
    n_tiles=len(tiles), library_types=len(lp['ids']),
    instances_total=int(len(TYa)), residue_nt_beads_total=int(np.diff(boff)[TYa].sum()), mrna_threads=int(len(MRT) if MRT is not None else 0), mrna_beads=int(len(MRP)),
    chromatin=dict(nucleosomes_placed=int(len(pre['nuc_pos'])), nucleosomes_requested=int(n_nuc), fragments=56),
    npc=int(len(pre['npc_pos'])),
    placed_fraction_total=float(reg_tab.placed.sum() / reg_tab.requested.sum()),
    placed_fraction_outside_mitochondria=frac_nonmito,
    regions=reg_tab.reset_index().rename(columns={'index': 'region'}).round(4).to_dict(orient='records'),
    clash=clash if 'clash' in dir() else None,
    containment=contain, membrane_orientation=orient,
    density_mg_per_ml=mgml, mass_fraction_by_structure_source=srcm, instance_fraction_by_structure_source=srcn,
    acceptance=dict(placed_outside_mito_ge_0p99=frac_nonmito >= 0.99))
print([k for k in dir() if 'clash' in k.lower()])

mass by source {'AF': 0.6196, 'PDB': 0.3804} | instances by source {'AF': 0.8516, 'PDB': 0.1484} | PDB types 19
['clash', 'n_clash', 'n_clash_x']


In [50]:
# homo-oligomer PDB assemblies (phase 1; paralogs reuse the same entry)
OLIGO_PDB = {'TDH3': '3PYM', 'PDC1': '1QPB', 'CDC19': '1A3W', 'ADH1': '4W6Z', 'TPI1': '1NEY', 'GPM1': '1QHF', 'ENO1': '2AL1', 'AHP1': '4H86',
             'SOD1': '1F1G', 'HSP82': '2WER', 'IPP1': '1E9G', 'GND1': '2P4Q', 'TRR1': '3ITJ', 'HOM6': '1EBF', 'YDJ1': '1XAO', 'CYS3': '1N8P',
             'TKL1': '1GPU', 'KRS1': '7F6W'}
for a, b in {'TDH1': 'TDH3', 'TDH2': 'TDH3', 'ENO2': 'ENO1', 'ADH2': 'ADH1', 'PDC5': 'PDC1'}.items(): OLIGO_PDB[a] = OLIGO_PDB[b]
for k, v in OLIGO_PDB.items(): fixed[k + '_oligo'] = v
SRC2 = np.array([src_of(str(i))[0] for i in lp['ids']]); PDBID = np.array([src_of(str(i))[1] for i in lp['ids']])
src_all = SRC2[TYa]
srcm = (lambda s: (s / s.sum()).round(4).to_dict())(pd.Series(mass_all).groupby(src_all).sum())
srcn = pd.Series(src_all).value_counts(normalize=True).round(4).to_dict()
print('mass by source', srcm, '| instances by source', srcn, '| PDB types', int((SRC2 == 'PDB').sum()))
val['mass_fraction_by_structure_source'] = srcm; val['instance_fraction_by_structure_source'] = srcn
val['clash'] = {k: (float(v) if isinstance(v, (np.floating,)) else v) for k, v in clash.items()}
val['acceptance']['clash_rate_lt_1e-5'] = bool(clash['clash_rate'] < 1e-5)
val['unplaced_total'] = int(reg_tab.unplaced.sum())
import json
class NpEnc(json.JSONEncoder):
    def default(self, o):
        if isinstance(o, np.integer): return int(o)
        if isinstance(o, np.floating): return float(o)
        if isinstance(o, np.bool_): return bool(o)
        return super().default(o)
json.dump(val, open('/mnt/results/model/wholecell3d_validation.json', 'w'), indent=1, cls=NpEnc)
print(reg_tab[['requested', 'placed', 'unplaced', 'frac_placed']].astype({'requested': int, 'placed': int, 'unplaced': int}).round(3).to_string())

mass by source {'AF': 0.4883, 'PDB': 0.5117} | instances by source {'AF': 0.7708, 'PDB': 0.2292} | PDB types 42
                requested    placed  unplaced  frac_placed
cytosol          18743624  18741626      1998        1.000
nucleoplasm       1352011   1351894       117        1.000
nucleolus          607466    586273     21193        0.965
ER_lumen           272721    271023      1698        0.994
vacuole            156608    156607         1        1.000
mito_matrix       2110584   1157052    953532        0.548
mito_IMS           234509     99403    135106        0.424
golgi_endosome     318129    311117      7012        0.978
peroxisome         101749    101698        51        0.999
cell_wall          388856    388856         0        1.000
PM_peripheral      289133    289014       119        1.000
NE_peripheral       86437     86380        57        0.999
ER_peripheral        5070      4461       609        0.880
LD_peripheral       83613     79911      3702        0.956
mem

In [51]:
np.savez('/workspace/yc3d/libpack_meta.npz', ids=np.array([str(i) for i in lp['ids']]), BR=lp['BR'], src=SRC2, pdb=PDBID)
print(len(lp['ids']), lp.keys() if isinstance(lp, dict) else None, pre['li_npc'], pre['li_nuc'])
import inspect; print(inspect.signature(c3.min_mem_dist), inspect.signature(c3.label_points))

5483 dict_keys(['ids', 'st_off', 'STC', 'STR', 'BR', 'memb', 'nbeads']) 2639 5479
(out, skip) (P, G, pores, pidx, PT, PQ, PC)


## Phase 2 / Step 2: encode library, tiles and geometry for the WebGPU viewer

In [52]:
import sys, os, json, time, importlib, gzip, shutil
sys.path.insert(0, '/workspace/yc')
import numpy as np, pandas as pd
import cell3d as c3, tiles3d as t3, webenc as we
importlib.reload(we)
meta = dict(np.load('/workspace/yc3d/libpack_meta.npz'))
ids = [str(x) for x in meta['ids']]; NT = len(ids)
ing2 = pd.read_csv('/mnt/results/inventory/ingredients_v2.csv')
cell = c3.Cell.load('/workspace/yc3d/geometry.json')
tiles = we.tile_index_list(); tindex = {t: i for i, t in enumerate(tiles)}
pre = dict(np.load('/workspace/yc3d/prepass.npz'))
pp = np.concatenate([pre['npc_pos'], pre['nuc_pos']])
pre_tile = np.array([tindex[tuple(int(v) for v in np.floor(p / we.TILE))] for p in pp])
S = we._S
S.update(tiles=tiles, tset=set(tiles), cell=cell, BR=meta['BR'].astype(np.float64), pre_tile=pre_tile,
         pre_type=np.r_[np.full(len(pre['npc_pos']), int(pre['li_npc'])), np.full(len(pre['nuc_pos']), int(pre['li_nuc']))],
         pre_pos=pp.astype(np.float64), pre_q=np.concatenate([pre['npc_q'], pre['nuc_q']]).astype(np.float64),
         pre_reg=np.r_[np.full(len(pre['npc_pos']), we.R_NPC), np.full(len(pre['nuc_pos']), we.R_NUCLEOSOME)],
         tmp='/workspace/yc3d/web_tmp', out=we.OUT)
os.makedirs(S['tmp'], exist_ok=True); os.makedirs(f'{we.OUT}/tiles', exist_ok=True)
print(len(tiles), 'tiles;', NT, 'types; pre-pass', len(pp))
# small test then full stage 1
import multiprocessing as mp
t0 = time.time(); r = we._stage1(3000); print('one tile', r[:3], f'{time.time()-t0:.1f}s')
t0 = time.time()
with mp.get_context('fork').Pool(16) as pool:
    st1 = pool.map(we._stage1, range(len(tiles)), chunksize=8)
print(f'stage1 {time.time()-t0:.0f}s; instances {sum(x[1] for x in st1)/1e6:.3f}M; near-membrane {sum(x[2] for x in st1)/1e6:.3f}M')
cnt = {}
for x in st1:
    for k, c in zip(x[3], x[4]): cnt[int(k)] = cnt.get(int(k), 0) + int(c)
counts_by_type = np.zeros(NT, np.int64); region_counts = {}
for k, c in cnt.items():
    t, rg = divmod(k, 32); counts_by_type[t] += c; region_counts.setdefault(t, {})[rg] = c
print('total', counts_by_type.sum())

6416 tiles; 5483 types; pre-pass 73267


one tile (3000, 6427, 1320) 1.2s


stage1 5s; instances 25.214M; near-membrane 2.640M


total 25213823


In [53]:
t0 = time.time()
libinfo = we.encode_library(meta, ing2, counts_by_type, region_counts)
print(libinfo, f'{time.time()-t0:.0f}s')
for f in os.listdir(f'{we.OUT}/library'): print(f, os.path.getsize(f'{we.OUT}/library/{f}')/1e6, 'MB')

{'n_types': 5483, 'n_beads': 2470774, 'n_st': 82562, 'n_af_plddt': 5441, 'n_af': 5441} 47s
beads.bin 19.766192 MB
spheretree.bin 1.320992 MB
types.json 1.656392 MB


In [54]:
importlib.reload(we); S = we._S if we._S else S; we._S.update(S)
zn = np.load(f'{we.LIB}/nucleosome.npz')
links, mids = we.linker_chains(pre, zn['beads'].astype(np.float64), zn['kind'])
link_tile = np.array([tindex.get(tuple(int(v) for v in np.floor(p / we.TILE)), -1) for p in mids])
print(f'linker DNA segments {len(links)}, beads {sum(len(l) for l in links)}, unassigned {(link_tile<0).sum()}, length median {np.median([np.linalg.norm(np.diff(l,axis=0),axis=1).sum() for l in links]):.1f} nm')
we._S.update(links=links, link_tile=link_tile)
t0 = time.time()
with mp.get_context('fork').Pool(16) as pool:
    st2 = pool.map(we._stage2, range(len(tiles)), chunksize=4)
cols = ['ti', 'n_inst', 'n_ghost', 'n_mrna', 'n_mrna_beads', 'n_link', 'n_link_beads', 'n_vert', 'n_tri', 'raw_bytes', 'gz_bytes']
st2 = pd.DataFrame(st2, columns=cols)
print(f'stage2 {time.time()-t0:.0f}s')
print(st2[cols[1:]].sum().to_string())
print(f"raw {st2.raw_bytes.sum()/1e6:.0f} MB, gz {st2.gz_bytes.sum()/1e6:.0f} MB, max gz tile {st2.gz_bytes.max()/1e6:.2f} MB")

linker DNA segments 73094, beads 1816415, unassigned 0, length median 16.7 nm


stage2 30s
n_inst           25213823
n_ghost            932883
n_mrna              30000
n_mrna_beads     28758168
n_link              73094
n_link_beads      1816415
n_vert            4815458
n_tri             8906828
raw_bytes       603083384
gz_bytes        515760923
raw 603 MB, gz 516 MB, max gz tile 0.34 MB


In [55]:
from scipy.spatial.transform import Rotation as Rot
rng = np.random.default_rng(3)
perr, aerr, merr, lerr, verr = [], [], [], [], []
for ti in rng.choice(len(tiles), 40, replace=False):
    t = tiles[ti]; name = t3.tile_name(t)
    d = we.decode_tile(gzip.decompress(open(f'{we.OUT}/tiles/{t[0]}_{t[1]}_{t[2]}.bin.gz', 'rb').read()))
    z = np.load(f"{S['tmp']}/{name}.npz"); o = np.lexsort((z['region'], z['types']))
    ty, po, qu, rg, fl = d['inst']
    assert (ty == z['types'][o]).all() and (rg == z['region'][o]).all() and (fl == z['flags'][o]).all()
    perr.append(np.abs(po - z['pos'][o]).max(initial=0))
    if len(qu):
        q0 = z['quat'][o]; dots = np.abs((qu * q0 / np.linalg.norm(q0, axis=1, keepdims=True)).sum(1))
        aerr.append(np.degrees(2 * np.arccos(np.clip(dots, 0, 1))).max())
    r = np.load(f'{we.RAW}/{name}.npz'); mo = r['mrna_off']
    for i, c in enumerate(d['mrna']): merr.append(np.abs(c - r['mrna_pts'][mo[i]:mo[i+1]]).max())
    for c, li in zip(d['linker'], np.where(link_tile == ti)[0]): lerr.append(np.abs(c - links[li]).max())
    g = np.load(f'{we.GDIR}/{name}.npz')
    if len(g['verts']): verr.append(np.abs(d['verts'] - g['verts']).max()); assert (d['faces'] == g['faces']).all()
rt = dict(tiles_tested=40, max_pos_err_nm=float(max(perr)), max_rot_err_deg=float(max(aerr)), max_mrna_err_nm=float(max(merr)),
          max_linker_err_nm=float(max(lerr)) if lerr else None, max_mesh_err_nm=float(max(verr)))
print(rt); assert rt['max_pos_err_nm'] < 0.01 and rt['max_rot_err_deg'] < 0.1

{'tiles_tested': 40, 'max_pos_err_nm': 0.0027466239414479787, 'max_rot_err_deg': 0.003630688467290472, 'max_mrna_err_nm': 0.0025000000000545697, 'max_linker_err_nm': 0.004999344173754139, 'max_mesh_err_nm': 0.0027465401210520213}


In [56]:
# ---- tile index
labvol = pd.read_parquet('/workspace/yc3d/tile_labvol.parquet')
labmask = ((labvol.values > 0).astype(np.int64) << np.arange(labvol.shape[1])).sum(1)
st2s = st2.sort_values('ti').reset_index(drop=True)
index = dict(tile_nm=we.TILE, q_org=we.Q_ORG, q_range=we.Q_RANGE, mrna_q=we.MR_Q, link_q=we.LK_Q,
             label_names=c3.LABEL_NAMES, region_names=t3.REGION_NAMES + ['chromatin_nucleosome', 'NE_pore_NPC'],
             mem_names=c3.MEM_NAMES, label_columns=list(labvol.columns),
             tiles=[[*tiles[i], int(r.n_inst), int(r.n_ghost), int(r.n_mrna_beads), int(r.n_link_beads), int(r.n_vert), int(r.n_tri), int(r.gz_bytes), int(labmask[i])]
                    for i, r in st2s.iterrows()],
             tile_columns=['ix', 'iy', 'iz', 'n_inst', 'n_ghost', 'n_mrna_beads', 'n_link_beads', 'n_vert', 'n_tri', 'gz_bytes', 'label_mask'])
json.dump(index, open(f'{we.OUT}/tiles/index.json', 'w'), separators=(',', ':'))
# ---- type -> tiles (for fly-to-nearest copy)
tt = [[] for _ in range(NT)]
for i, t in enumerate(tiles):
    for ty in np.unique(np.load(f"{S['tmp']}/{t3.tile_name(t)}.npz")['types']): tt[ty].append(i)
offs = np.zeros(NT + 1, np.uint32); offs[1:] = np.cumsum([len(x) for x in tt])
with open(f'{we.OUT}/types_tiles.bin', 'wb') as f:
    f.write(offs.astype('<u4').tobytes()); f.write(np.concatenate([np.array(x, '<u2') for x in tt]).tobytes())
print('types_tiles entries', offs[-1], os.path.getsize(f'{we.OUT}/types_tiles.bin') / 1e6, 'MB')
# ---- geometry for WGSL: primitives + uniform acceleration grid (prims whose bounding sphere + 12 nm touches a cell)
geo = json.load(open('/workspace/yc3d/geometry.json'))
GN, EXT = 32, 2200.0; h = 2 * EXT / GN
bc, br = cell.bc, cell.br
pc = cell.G[3:6] + cell.pores * (cell.G[6] - 0.5 * c3.PERI_W); pr = c3.PORE_R + c3.PERI_W + 12
cells_p, cells_q = [], []
for ix in range(GN):
    for iy in range(GN):
        for iz in range(GN):
            lo = -EXT + h * np.array([ix, iy, iz]); hi = lo + h
            q = np.clip(bc, lo, hi); cells_p.append(np.where(((q - bc) ** 2).sum(1) <= (br + 12) ** 2)[0])
            q = np.clip(pc, lo, hi); cells_q.append(np.where(((q - pc) ** 2).sum(1) <= pr ** 2)[0])
goff = np.zeros(GN ** 3 + 1, np.int64); goff[1:] = np.cumsum([len(a) for a in cells_p])
qoff = np.zeros(GN ** 3 + 1, np.int64); qoff[1:] = np.cumsum([len(a) for a in cells_q])
print('grid prim refs', goff[-1], 'max/cell', max(len(a) for a in cells_p), '| pore refs', qoff[-1], 'max/cell', max(len(a) for a in cells_q))
gweb = dict(G=geo['G'], pores=cell.pores.round(6).ravel().tolist(), PT=geo['PT'], PQ=np.round(cell.PQ, 4).ravel().tolist(), PC=geo['PC'],
            consts=geo['consts'], label_names=c3.LABEL_NAMES, mem_names=c3.MEM_NAMES, npc_pos=geo['npc_pos'], mito_segments=geo['mito_segments'],
            grid=dict(n=GN, ext=EXT, prim_off=goff.tolist(), prim_idx=np.concatenate(cells_p).tolist(), pore_off=qoff.tolist(), pore_idx=np.concatenate(cells_q).tolist()))
json.dump(gweb, open(f'{we.OUT}/geometry.json', 'w'), separators=(',', ':'))
print('geometry.json', os.path.getsize(f'{we.OUT}/geometry.json') / 1e6, 'MB')

types_tiles entries 3953305 7.928546 MB


grid prim refs 27586 max/cell 18 | pore refs 1189 max/cell 4
geometry.json 0.465883 MB


In [57]:
import fast_simplification as fs
Vs = {m: [] for m in range(10)}; Fs = {m: [] for m in range(10)}; nv = {m: 0 for m in range(10)}
for t in tiles:
    g = np.load(f'{we.GDIR}/{t3.tile_name(t)}.npz')
    if not len(g['faces']): continue
    V, F, FM = g['verts'], g['faces'], g['face_mem']
    for m in np.unique(FM):
        f = F[FM == m]; u, inv = np.unique(f.ravel(), return_inverse=True)
        Vs[m].append(V[u]); Fs[m].append(inv.reshape(-1, 3) + nv[m]); nv[m] += len(u)
print('membranes with mesh faces:', [c3.MEM_NAMES[m] for m in range(10) if Fs[m]])
mesh_out = []; tot = 0
t0 = time.time()
for m in [m for m in range(10) if Fs[m]]:
    V = np.concatenate(Vs[m]).astype(np.float64); F = np.concatenate(Fs[m])
    key = np.round(V / 0.01).astype(np.int64)
    _, first, inv = np.unique(key, axis=0, return_index=True, return_inverse=True)
    V2 = V[first]; F2 = inv.ravel()[F]
    F2 = F2[(F2[:, 0] != F2[:, 1]) & (F2[:, 1] != F2[:, 2]) & (F2[:, 0] != F2[:, 2])]
    red = 0.95 if m in (0, 3) else 0.9
    Vd, Fd = fs.simplify(V2.astype(np.float32), F2.astype(np.int32), target_reduction=red)
    mesh_out.append((m, Vd.astype('<f4'), Fd.astype('<u4'))); tot += len(Fd)
    print(c3.MEM_NAMES[m], len(F2), '->', len(Fd), f'{time.time()-t0:.0f}s')
with gzip.open(f'{we.OUT}/organelle_meshes.bin.gz', 'wb', compresslevel=9) as f:
    hdr = np.zeros(1 + 3 * len(mesh_out), np.uint32); hdr[0] = len(mesh_out)
    for i, (m, Vd, Fd) in enumerate(mesh_out): hdr[1 + 3 * i: 4 + 3 * i] = [m, len(Vd), len(Fd)]
    f.write(hdr.tobytes())
    for m, Vd, Fd in mesh_out: f.write(Vd.tobytes()); f.write(Fd.tobytes())
print('faces total', tot, os.path.getsize(f'{we.OUT}/organelle_meshes.bin.gz') / 1e6, 'MB')

membranes with mesh faces: ['PM', 'ER', 'NE', 'VAC', 'MITO_OM', 'MITO_IM', 'GOLGI', 'PEROX', 'LD']


PM 2466296 -> 123314 4s


ER 2437882 -> 243788 8s


NE 855570 -> 85556 9s


VAC 465080 -> 23254 10s


MITO_OM 581876 -> 58186 11s


MITO_IM 1341632 -> 134162 15s


GOLGI 413336 -> 50392 16s
PEROX 68894 -> 6888 16s


LD 273664 -> 27366 16s


faces total 752906 6.707719 MB


In [58]:
def unit(v): v = np.asarray(v, float); return v / np.linalg.norm(v)
def perp(a, hint=(0.3, 0.5, 0.81)):
    p = np.cross(a, hint); return unit(p)
li = {k: ids.index(k) for k in ['Pma1_hexamer', 'ribosome_80S', 'nucleosome', 'NPC']}
tcount = {k: np.zeros(len(tiles), int) for k in li}
for i, t in enumerate(tiles):
    ty = np.load(f"{S['tmp']}/{t3.tile_name(t)}.npz")['types']
    for k, v in li.items(): tcount[k][i] = (ty == v).sum()
lv = labvol.values; lcol = list(labvol.columns)
def tc(i): return np.array(tiles[i], float) * 200 + 100
bm = []
G = cell.G; cn = G[3:6]; vc = np.array([985., -170., -110.])
n0 = unit(np.cross(cn, vc))
bm.append(dict(name='Whole cell', caption='Equatorial section through nucleus and vacuole. 25.2 million macromolecules, every one a real PDB or AlphaFold structure.',
               target=[0, 0, 0], normal=n0, up=unit(cn), width=5200, slab=60))
# cell wall: surface point away from nucleus/vacuole
u = unit(-cn - vc); p = u * (G[0] - 55)
bm.append(dict(name='Cell wall', caption='Mannoprotein brush (outer 35 nm) over the beta-glucan layer, plasma membrane below.', target=p, normal=perp(u), up=u, width=260, slab=40))
# Pma1-rich PM (face-on from the cytoplasm, tile without cER)
cer_col = lcol.index('cER_lumen') if 'cER_lumen' in lcol else None
cand = [i for i in np.argsort(-tcount['Pma1_hexamer']) if (cer_col is None or lv[i, cer_col] == 0)][:1]
u = unit(tc(cand[0])); p = u * (G[2] - 18)
bm.append(dict(name='Plasma membrane (face-on)', caption='Cytoplasmic face of the plasma membrane: Pma1 H+-ATPase hexamers among ~1,000 other membrane proteins; lipids generated around them.',
               target=p, normal=u, up=perp(u), width=220, slab=30))
# cortical ER patch section
cer = [i for i, c in enumerate(cell.PC) if c == c3.C_CER]
q = cell.PQ[cer[0]]; u = unit(q[:3]); p = u * q[5]
bm.append(dict(name='Cortical ER', caption='Cortical ER sheet 27 nm beneath the plasma membrane, with Sec61 translocons and ER-bound ribosomes.', target=p, normal=perp(u), up=u, width=320, slab=40))
# ribosome-crowded cytoplasm
cyto = lcol.index('cytosol')
i = [i for i in np.argsort(-tcount['ribosome_80S']) if lv[i, cyto] > 0.99 * lv[i].sum()][0]
bm.append(dict(name='Cytoplasm', caption='Crowded cytoplasm (~89 mg/mL): 80S ribosomes (pink), glycolytic enzymes, tRNAs and mRNA threads.', target=tc(i), normal=unit([0.2, -0.4, 0.89]), up=unit([0, 1, 0.45]), width=160, slab=30))
# nuclear pore: pore closest to cell surface
pcen = cn + cell.pores * (G[6] - 0.5 * c3.PERI_W)
k = int(np.argmax(np.linalg.norm(pcen, axis=1)))
ax = unit(cell.pores[k])
bm.append(dict(name='Nuclear pore complex', caption='NPC (PDB 7N9F) seated in a toroidal pore membrane joining the outer and inner nuclear membranes.', target=pcen[k], normal=perp(ax), up=ax, width=260, slab=50))
# chromatin
nucp = lcol.index('nucleoplasm')
i = [i for i in np.argsort(-tcount['nucleosome']) if lv[i, nucp] > 0.95 * lv[i].sum()][0]
bm.append(dict(name='Chromatin', caption='Nucleoplasm: nucleosome chains (73,150 nucleosomes, linker DNA) with RNA polymerase II and nuclear proteins.', target=tc(i), normal=unit([0.5, 0.3, -0.81]), up=unit([0, 1, 0.37]), width=200, slab=40))
# nucleolus boundary
nucl = lcol.index('nucleolus')
candl = [i for i in range(len(tiles)) if lv[i, nucl] > 0.3 * lv[i].sum() and lv[i, nucp] > 0.3 * lv[i].sum()]
i = candl[0]; uu = unit(G[7:10] - cn)
bm.append(dict(name='Nucleolus', caption='Nucleolar crescent (ribosome assembly factors, snoRNPs, Pol I) bordering the nucleoplasm.', target=tc(i), normal=perp(uu), up=uu, width=450, slab=40))
# mitochondrion: segment midpoint, plane containing the tube axis
seg = np.array(geo['mito_segments'][len(geo['mito_segments']) // 2]); a = unit(seg[1] - seg[0]); p = seg.mean(0)
bm.append(dict(name='Mitochondrion', caption='Mitochondrial tubule: outer and inner membranes, lamellar cristae and the protein-dense matrix (~110 mg/mL).', target=p, normal=perp(a), up=a, width=420, slab=40))
# vacuole membrane, toward cell centre
u = unit(-vc); p = vc + u * 889.0
bm.append(dict(name='Vacuole', caption='Vacuolar membrane with V-ATPases; the vacuole lumen is dilute (~5 mg/mL).', target=p, normal=perp(u), up=u, width=300, slab=40))
# lipid droplet
ld = [i for i, c in enumerate(cell.PC) if c == c3.C_LD][0]; q = cell.PQ[ld]
bm.append(dict(name='Lipid droplet', caption='Lipid droplet: neutral-lipid core with a phospholipid monolayer and peripheral proteins, touching ER.', target=q[:3] + unit([0, 0, 1]) * 0, normal=unit([0, 0, 1]), up=unit([0, 1, 0]), width=560, slab=40))
# golgi
go = [i for i, c in enumerate(cell.PC) if c == c3.C_GOLGI][0]; q = cell.PQ[go]
bm.append(dict(name='Golgi / endosomes', caption='Golgi cisternae and endosomes with their membrane and lumenal proteins.', target=q[:3], normal=unit([0, 0, 1]), up=unit([0, 1, 0]), width=500, slab=40))
for b in bm:
    for k in ('target', 'normal', 'up'): b[k] = [round(float(x), 3) for x in b[k]]
    p = np.array(b['target']); print(b['name'], c3.LABEL_NAMES[cell.label(p)[0]], np.round(p))
json.dump(bm, open(f'{we.OUT}/bookmarks.json', 'w'), indent=1)

Whole cell golgi_endosome [0. 0. 0.]
Cell wall wall_glucan [ 459.  917. 1834.]
Plasma membrane (face-on) cytosol [  694.   892. -1685.]
Cortical ER cER_lumen [  -23. -1978.   354.]
Cytoplasm cytosol [-1300.  1100.   700.]
Nuclear pore complex pore_channel [-1856.   250.   122.]
Chromatin nucleoplasm [-300. -100.  100.]
Nucleolus nucleoplasm [-1500.  -300.  -100.]
Mitochondrion mito_IMS [ -860. -1019.  1070.]
Vacuole cytosol [114. -20. -13.]
Lipid droplet lipid_droplet [-139.   48. -733.]
Golgi / endosomes golgi_endosome [ 557. 1247.  953.]


In [59]:
# mesh face normal orientation vs membrane SDF gradient (+ = cytoplasmic side)
res = {}
for t in [tiles[i] for i in rng.choice(len(tiles), 300, replace=False)]:
    g = np.load(f'{we.GDIR}/{t3.tile_name(t)}.npz')
    if not len(g['faces']): continue
    V, F, FM = g['verts'].astype(float), g['faces'], g['face_mem']
    for j in rng.choice(len(F), min(20, len(F)), replace=False):
        a, b, c = V[F[j]]; nrm = np.cross(b - a, c - a); nrm /= np.linalg.norm(nrm); ctr = (a + b + c) / 3; m = FM[j]
        lo = np.array(t) * 200.; A = cell.args(lo, lo + 200)
        out = np.zeros(c3.NMEM); c3.mem_sdfs(*(ctr + 0.5 * nrm), *A, out); s1 = out[m]
        c3.mem_sdfs(*(ctr - 0.5 * nrm), *A, out); s0 = out[m]
        res.setdefault(c3.MEM_NAMES[m], []).append(np.sign(s1 - s0))
print({k: (np.mean(v), len(v)) for k, v in res.items()})
# wall polysaccharide budgets (phase-1 assumptions: protein 45% of dry mass, wall 20% of dry mass, Klis 2002 composition)
prot_da = float((ing2.copies_per_cell * ing2.mass_da).sum())
dry = prot_da / 0.45; wall = dry * 0.20
glc_res = wall * (0.525 + 0.075) / 162.14; man_res = wall * 0.375 * 0.85 / 162.14
vol_glucan = labvol['wall_glucan'].sum(); vol_mannan = labvol['wall_mannan'].sum()
n_wallprot_mannan = 0
for i, t in enumerate(tiles):
    if not (lv[i, lcol.index('wall_mannan')] > 0): continue
    z = np.load(f"{S['tmp']}/{t3.tile_name(t)}.npz")
    k = z['region'] == t3.REGION_NAMES.index('cell_wall')
    n_wallprot_mannan += int((np.linalg.norm(z['pos'][k], axis=1) > cell.G[1] - 5).sum())
wallp = dict(dry_mass_pg=dry * 1.66054e-12, wall_mass_pg=wall * 1.66054e-12, glucan_glucose=glc_res, mannose=man_res,
             glucan_beads_per_nm3=glc_res / 36 / vol_glucan, glucan_layer_um3=vol_glucan / 1e9, mannan_layer_um3=vol_mannan / 1e9,
             wall_proteins_in_mannan_layer=n_wallprot_mannan, mannan_beads_per_protein=man_res / 3 / max(n_wallprot_mannan, 1))
print(wallp)
index['wall'] = wallp
for i, row in enumerate(index['tiles']):
    row += [round(float(labvol.iloc[i]['wall_glucan']), 1), round(float(labvol.iloc[i]['wall_mannan']), 1)]
index['tile_columns'] += ['vol_glucan_nm3', 'vol_mannan_nm3']
json.dump(index, open(f'{we.OUT}/tiles/index.json', 'w'), separators=(',', ':'))

{'ER': (np.float64(1.0), 634), 'PM': (np.float64(1.0), 1375), 'GOLGI': (np.float64(1.0), 335), 'MITO_OM': (np.float64(1.0), 172), 'MITO_IM': (np.float64(1.0), 152), 'NE': (np.float64(1.0), 314), 'VAC': (np.float64(1.0), 263), 'LD': (np.float64(1.0), 159), 'PEROX': (np.float64(1.0), 80)}


{'dry_mass_pg': 5.998810125487268, 'wall_mass_pg': 1.1997620250974537, 'glucan_glucose': 2673664142.657704, 'mannose': 1420384075.7869053, 'glucan_beads_per_nm3': np.float64(0.01814673954131497), 'glucan_layer_um3': np.float64(4.092660736), 'mannan_layer_um3': np.float64(2.012096512), 'wall_proteins_in_mannan_layer': 248068, 'mannan_beads_per_protein': 1908.5950569829045}
